In [3]:
'''
Ανάλυση χρονοσειρων με Denoising μέσω PyTorch 1D Convolutional Neural Network (CNNs) και Time Delay Neural Networks TDNNs
 
'''
# Εισαγωγή των απαραίτητων βιβλιοθηκών

import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns 
from scipy import signal 
from scipy.stats import norm 
import pandas as pd 
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

import torch 
import torch.nn as nn 
import torch.optim as optim 
from torch.utils.data import Dataset, DataLoader 
import torch.nn.functional as F
import yfinance as yf 


In [4]:
# Device setup 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

#Seed για αναπαραγωγιμότητα
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

Using device: cuda


In [ ]:
class TimeSeriesGenerator: 
    @staticmethod
    def gravitational_wave_signal(duration=2.0, sampling_rate=5000, frequency_start=20,frequency_end=250):
        t = np.linspace(0, duration, int(duration * sampling_rate), endpoint=False)
        chirp = signal.chirp(t, frequency_start, duration, frequency_end, method='linear')
        envelope = np.exp(-((t - duration/2)**2) / (2 * (duration/5)**2))
        signal_clean = chirp * envelope
        
        n_spikes = 10
        spike_positions = np.random.random_integers(0, len(signal_clean), n_spikes)
        spike_amplitudes = np.random.uniform(0.5, 2, n_spikes)
        for pos, amp in zip(spike_positions, spike_amplitudes):
            signal_clean[max(0, pos-5):min(len(signal_clean), pos+5)] += amp * signal.windows.hann(10)
        return t, signal_clean
    @staticmethod
    def economic_signal(length=1000, volatility=0.02):
        """
        Σήμα οικονομικών δεδομένων (log-returns)
        Με περιόδους κρίσης (regime shifts)
        """
        # Drift
        drift = 0.0001
        
        # Δημιουργία log-returns με GBM
        returns = np.zeros(length)
        returns[0] = 0
        
        # Regime shifts (κρίσεις)
        regime = np.ones(length)
        crisis_periods = [
            (100, 200, 3.0),   # Κρίση 1: χρόνος, διάρκεια, σοβαρότητα
            (400, 150, 2.5),   # Κρίση 2
            (700, 120, 2.0)    # Κρίση 3
        ]
        
        for start, duration, severity in crisis_periods:
            regime[start:start+duration] = severity
        
        for i in range(1, length):
            shock = np.random.normal(drift, volatility * regime[i])
            returns[i] = returns[i-1] + shock
        
        # Cumulative sum για τιμές
        prices = np.cumsum(returns)
        
        return np.arange(length), prices
    @staticmethod
    def add_noise(signal_clean, snr_db=10):
        """
        Add Gaussian noise with a specific SNR (Signal-to-Noise Ratio) in dB.
        SNR (dB) = 10 * log10(P_signal / P_noise)
        """
        signal_power = np.mean(signal_clean ** 2)
        noise_power = signal_power / (10 ** (snr_db / 10))
        noise = np.random.normal(0, np.sqrt(noise_power), len(signal_clean))
        signal_noisy = signal_clean + noise
        return signal_noisy
    @staticmethod
    def create_dataset(signal_clean, window_size=64, stride=8):
        X, y= [], []
        
        for i in range(0, len(signal_clean) - window_size, stride):
            X.append(signal_clean[i:i+window_size])
            y.append(signal_clean[i+window_size])
        return np.array(X), np.array(y)

class Conv1DAutoencoder(nn.Module):
    """
    1D Convolutional Autoencoder για denoising
    """
    
    def __init__(self, input_length=64, latent_dim=16):
        super(Conv1DAutoencoder, self).__init__()
        
        self.input_length = input_length
        self.latent_dim = latent_dim
        
        # Encoder
        self.enc_conv1 = nn.Conv1d(1, 32, kernel_size=5, padding=2, stride=1)
        self.enc_bn1 = nn.BatchNorm1d(32)
        self.enc_pool1 = nn.MaxPool1d(kernel_size=2)
        
        self.enc_conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2, stride=1)
        self.enc_bn2 = nn.BatchNorm1d(64)
        self.enc_pool2 = nn.MaxPool1d(kernel_size=2)
        
        self.enc_conv3 = nn.Conv1d(64, 128, kernel_size=3, padding=1, stride=1)
        self.enc_bn3 = nn.BatchNorm1d(128)
        
        # Bottleneck
        bottleneck_size = 128 * (input_length // 4)
        self.bottleneck_fc1 = nn.Linear(bottleneck_size, latent_dim)
        self.bottleneck_fc2 = nn.Linear(latent_dim, bottleneck_size)
        
        # Decoder
        self.dec_conv1 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.dec_bn1 = nn.BatchNorm1d(128)
        self.dec_up1 = nn.Upsample(scale_factor=2, mode='nearest')
        
        self.dec_conv2 = nn.Conv1d(128, 64, kernel_size=5, padding=2)
        self.dec_bn2 = nn.BatchNorm1d(64)
        self.dec_up2 = nn.Upsample(scale_factor=2, mode='nearest')
        
        self.dec_conv3 = nn.Conv1d(64, 32, kernel_size=5, padding=2)
        self.dec_bn3 = nn.BatchNorm1d(32)
        
        self.dec_conv4 = nn.Conv1d(32, 1, kernel_size=5, padding=2)
    
    def encode(self, x):
        # x: (batch, 1, window_size)
        x = F.relu(self.enc_bn1(self.enc_conv1(x)))
        x = self.enc_pool1(x)
        
        x = F.relu(self.enc_bn2(self.enc_conv2(x)))
        x = self.enc_pool2(x)
        
        x = F.relu(self.enc_bn3(self.enc_conv3(x)))
        
        x = x.view(x.size(0), -1)
        x = F.relu(self.bottleneck_fc1(x))
        
        return x
    
    def decode(self, z):
        # z: (batch, latent_dim)
        x = F.relu(self.bottleneck_fc2(z))
        x = x.view(x.size(0), 128, -1)
        
        x = F.relu(self.dec_bn1(self.dec_conv1(x)))
        x = self.dec_up1(x)
        
        x = F.relu(self.dec_bn2(self.dec_conv2(x)))
        x = self.dec_up2(x)
        
        x = F.relu(self.dec_bn3(self.dec_conv3(x)))
        x = self.dec_conv4(x)
        
        return x
    
    def forward(self, x):
        z = self.encode(x)
        x_recon = self.decode(z)
        return x_recon
    
class TDNN(nn.Module):
    """
    Time Delay Neural Network (TDNN)
    Χρησιμοποιεί temporal context windows για feature extraction
    """
    
    def __init__(self, input_length=64, time_delays=(1, 2, 4, 8)):
        super(TDNN, self).__init__()
        
        self.input_length = input_length
        self.time_delays = time_delays
        
        # Number of delayed features = 1 (original) + len(time_delays)
        n_delayed_features = 1 + len(time_delays)
        
        # Dense layers
        self.fc1 = nn.Linear(input_length * n_delayed_features, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.dropout1 = nn.Dropout(0.2)
        
        self.fc2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.dropout2 = nn.Dropout(0.2)
        
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, input_length)
    
    def forward(self, x):
        # x: (batch, 1, window_size)
        batch_size = x.size(0)
        x = x.squeeze(1)  # (batch, window_size)
        
        # Create time-delayed features
        delayed_features = [x]  # Original signal
        
        for delay in self.time_delays:
            if delay > 0:
                # Shift signal by delay and pad with zeros
                delayed = torch.cat([
                    torch.zeros(batch_size, delay, device=x.device),
                    x[:, :-delay]
                ], dim=1)
                delayed_features.append(delayed)
        
        # Concatenate all time-delayed versions
        x_delayed = torch.cat(delayed_features, dim=1)  # (batch, window_size * n_delayed)
        
        # Dense layers
        x = F.relu(self.bn1(self.fc1(x_delayed)))
        x = self.dropout1(x)
        
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.dropout2(x)
        
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        
        # Reshape to (batch, 1, window_size) for compatibility
        x = x.unsqueeze(1)
        
        return x
class DenoisingDataset(Dataset):
    """PyTorch Dataset για denoising"""
    
    def __init__(self, X_noisy, y_clean):
        """
        Args:
            X_noisy: (N, window_size) noisy signals
            y_clean: (N, window_size) clean signals
        """
        self.X_noisy = torch.FloatTensor(X_noisy)
        self.y_clean = torch.FloatTensor(y_clean)
    
    def __len__(self):
        return len(self.X_noisy)
    
    def __getitem__(self, idx):
        # Reshape to (1, window_size) for 1D convolution
        x = self.X_noisy[idx].unsqueeze(0)
        y = self.y_clean[idx].unsqueeze(0)
        return x, y

In [ ]:
def train_denoising_model(model, train_loader, val_loader, epochs=30, 
                         model_name="Model", device=device):
    """Εκπαίδευση μοντέλου denoising με PyTorch"""
    
    print(f"\n{'='*80}")
    print(f"ΕΚΠΑΙΔΕΥΣΗ ΜΟΝΤΕΛΟΥ: {model_name}")
    print(f"{'='*80}")
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr = 0.001)
    criterion = nn.MSELoss()
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    best_val_loss = float('inf')
    patience = 10
    patience_counter = 0
    train_losses, val_losses = [], []
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for batch_noisy, batch_clean in train_loader:
                batch_noisy, batch_clean = batch_noisy.to(device), batch_clean.to(device)
                optimizer.zero_grad()
                outputs = model(batch_noisy)
                loss = criterion(outputs, batch_clean)
                loss.backward()
                optimizer.step()
                train_loss += loss.item() 
        train_loss /= len(train_loader)
        train_losses.append(train_loss)
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_noisy, batch_clean in val_loader:
                batch_noisy, batch_clean = batch_noisy.to(device), batch_clean.to(device)
                outputs = model(batch_noisy)
                loss = criterion(outputs, batch_clean)
                val_loss += loss.item()
        val_loss /= len(val_loader)
        val_losses.append(val_loss)
        scheduler.step(val_loss)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}")
    
    print(f"✓ Εκπαίδευση ολοκληρώθηκε")
    print(f"  Final training loss: {train_losses[-1]:.6f}")
    print(f"  Final validation loss: {val_losses[-1]:.6f}")

    return train_losses, val_losses

In [ ]:
def evaluate_denoiser(model, test_loader, model_name="model", device=device):
    """Αξιολόγηση μοντέλου"""
    
    print(f"\n{'='*60}")
    print(f"ΑΞΙΟΛΟΓΗΣΗ: {model_name}")
    print(f"{'='*60}")
    
    model.eval()
    
    all_noisy = []
    all_clean = []
    all_denoised = []
    
    with torch.no_grad():
        for batch_noisy, batch_clean in test_loader:
            batch_noisy, batch_clean = batch_noisy.to(device), batch_clean.to(device)
            outputs = model(batch_noisy)
            all_noisy.append(batch_noisy.cpu().numpy())
            all_clean.append(batch_clean.cpu().numpy())
            all_denoised.append(outputs.cpu().numpy())
    X_test_noisy = np.concatenate(all_noisy, axis=0)
    X_test_clean = np.concatenate(all_clean, axis=0)
    X_test_denoised = np.concatenate(all_denoised, axis=0)
    
    # Metrics
    mse_noisy = np.mean((X_test_noisy - X_test_clean) ** 2)
    mse_denoised = np.mean((X_test_denoised - X_test_clean) ** 2)
    mae_noisy = np.mean(np.abs(X_test_noisy - X_test_clean))
    mae_denoised = np.mean(np.abs(X_test_denoised - X_test_clean))
    
    # SNR improvement
    snr_noise = 10 * np.log10(np.mean(X_test_clean ** 2) / mse_noisy + 1e-10)
    snr_denoised = 10 * np.log10(np.mean(X_test_clean ** 2) / mse_denoised + 1e-10)
    snr_improvement = snr_denoised - snr_noise
    
    print(f"MSE (noisy):     {mse_noisy:.6f}")
    print(f"MSE (denoised):  {mse_denoised:.6f}")
    print(f"Βελτίωση MSE:    {(1 - mse_denoised/mse_noisy)*100:.2f}%")
    print(f"\nMAE (noisy):     {mae_noisy:.6f}")
    print(f"MAE (denoised):  {mae_denoised:.6f}")
    print(f"Βελτίωση MAE:    {(1 - mae_denoised/mae_noisy)*100:.2f}%")
    print(f"\nSNR (noisy):     {snr_noise:.2f} dB")
    print(f"SNR (denoised):  {snr_denoised:.2f} dB")
    print(f"SNR Improvement: {snr_improvement:.2f} dB")
    
    return {
        'mse_noisy': mse_noisy,
        'mse_denoised': mse_denoised,
        'mae_noisy': mae_noisy,
        'mae_denoised': mae_denoised,
        'snr_noisy': snr_noise,
        'snr_denoised': snr_denoised,
        'snr_improvement': snr_improvement,
        'predictions': X_test_denoised
    }
    
def predict_full_signal(model, signal_noisy, window_size=64, stride=8, device=device):
    """Prediction σε ολόκληρο το σήμα"""
    model.eval()
    predictions = np.zeros_like(signal_noisy)
    counts = np.zeros_like(signal_noisy)
    
    with torch.no_grad():
        for i in range(0, len(signal_noisy) - window_size, stride):
            window = signal_noisy[i:i + window_size]
            window_tensor = torch.FloatTensor(window).unsqueeze(0).unsqueeze(0).to(device)
            
            output = model(window_tensor)
            output_np = output.squeeze().cpu().numpy()
            
            predictions[i:i + window_size] += output_np
            counts[i:i + window_size] += 1
    
    # Average overlapping regions
    predictions = np.divide(predictions, counts, where=counts > 0, out=predictions)
    
    return predictions

In [ ]:
def detect_events(signal_clean, signal_noisy, signal_denoised, threshold_std=2.0):
    """Ανίχνευση σημαντικών συμβάντων (events)"""
    
    results = {
        'clean': [],
        'noisy': [],
        'denoised': []
    }
    
    signal_clean = np.squeeze(np.array(signal_clean))
    signal_noisy = np.squeeze(np.array(signal_noisy))
    signal_denoised = np.squeeze(np.array(signal_denoised))
    
    for name, sig in [('clean', signal_clean), ('noisy', signal_noisy), 
                      ('denoised', signal_denoised)]:
        
        velocity = np.gradient(sig)
        acceleration = np.gradient(velocity)
        
        threshold = threshold_std * np.std(acceleration)
        event_mask = np.abs(acceleration) > threshold
        event_indices = np.where(event_mask)[0]
        
        event_clusters = []
        if len(event_indices) > 0:
            current_cluster = [event_indices[0]]
            for idx in event_indices[1:]:
                if idx - current_cluster[-1] <= 5:
                    current_cluster.append(idx)
                else:
                    if len(current_cluster) > 0:
                        event_clusters.append(current_cluster[:])
                    current_cluster = [idx]
            if len(current_cluster) > 0:
                event_clusters.append(current_cluster[:])
        
        for cluster in event_clusters:
            cluster_arr = np.array(cluster, dtype=int)
            try:
                event_data = {
                    'center': float(np.mean(cluster_arr)),
                    'duration': int(len(cluster_arr)),
                    'max_acceleration': float(np.max(np.abs(acceleration[cluster_arr]))),
                    'max_velocity': float(np.max(np.abs(velocity[cluster_arr]))),
                    'amplitude': float(np.max(sig[cluster_arr]) - np.min(sig[cluster_arr]))
                }
                results[name].append(event_data)
            except:
                pass
    
    return results

In [ ]:
def main():
    
    # ========== 1. ΔΗΜΙΟΥΡΓΙΑ ΔΕΔΟΜΕΝΩΝ ==========
    print("\n" + "="*80)
    print("ΦΑΣΗ 1: ΔΗΜΙΟΥΡΓΙΑ ΣΥΝΘΕΤΙΚΩΝ ΔΕΔΟΜΕΝΩΝ")
    print("="*80)
    
    # Gravitational wave signal
    print("\n[1.1] Δημιουργία σήματος βαρυτικών κυμάτων...")
    t_gw, signal_gw_clean = TimeSeriesGenerator.gravitational_wave_signal(
        duration=2.0, sampling_rate=4000
    )
    signal_gw_noisy = TimeSeriesGenerator.add_noise(signal_gw_clean, snr_db=5)
    print(f"✓ Signal length: {len(signal_gw_clean)} samples")
    print(f"  Clean signal power: {np.mean(signal_gw_clean**2):.6f}")
    print(f"  Noise power: {np.mean((signal_gw_noisy - signal_gw_clean)**2):.6f}")
    
    # Economic signal
    print("\n[1.2] Δημιουργία οικονομικού σήματος (S&P 500)...")
    t_econ, signal_econ_clean = TimeSeriesGenerator.economic_signal(length=2000)
    signal_econ_noisy = TimeSeriesGenerator.add_noise(signal_econ_clean, snr_db=15)
    print(f"✓ Signal length: {len(signal_econ_clean)} samples")
    print(f"  Range: [{signal_econ_clean.min():.2f}, {signal_econ_clean.max():.2f}]")
    
    # ========== 2. ΠΡΟΕΤΟΙΜΑΣΙΑ DATASETS ==========
    print("\n" + "="*80)
    print("ΦΑΣΗ 2: ΠΡΟΕΤΟΙΜΑΣΙΑ DATASETS")
    print("="*80)
    
    window_size = 64
    batch_size = 32
    
    # Gravitational wave dataset
    print(f"\n[2.1] Δημιουργία dataset για GW signal (window_size={window_size})...")
    X_gw_clean, y_gw_clean = TimeSeriesGenerator.create_dataset(
        signal_gw_clean, window_size=window_size, stride=8
    )
    X_gw_noisy, _ = TimeSeriesGenerator.create_dataset(
        signal_gw_noisy, window_size=window_size, stride=8
    )
    
    # Split train/val/test
    n_train = int(0.6 * len(X_gw_clean))
    n_val = int(0.2 * len(X_gw_clean))
    
    X_gw_train_clean = X_gw_clean[:n_train]
    X_gw_train_noisy = X_gw_noisy[:n_train]
    X_gw_val_clean = X_gw_clean[n_train:n_train+n_val]
    X_gw_val_noisy = X_gw_noisy[n_train:n_train+n_val]
    X_gw_test_clean = X_gw_clean[n_train+n_val:]
    X_gw_test_noisy = X_gw_noisy[n_train+n_val:]
    
    print(f"✓ Train: {len(X_gw_train_clean)}, Val: {len(X_gw_val_clean)}, Test: {len(X_gw_test_clean)}")
    
    # Create PyTorch DataLoaders
    train_ds_gw = DenoisingDataset(X_gw_train_noisy, X_gw_train_clean)
    val_ds_gw = DenoisingDataset(X_gw_val_noisy, X_gw_val_clean)
    test_ds_gw = DenoisingDataset(X_gw_test_noisy, X_gw_test_clean)
    
    train_loader_gw = DataLoader(train_ds_gw, batch_size=batch_size, shuffle=True)
    val_loader_gw = DataLoader(val_ds_gw, batch_size=batch_size, shuffle=False)
    test_loader_gw = DataLoader(test_ds_gw, batch_size=batch_size, shuffle=False)
    
    # Economic signal dataset
    print(f"\n[2.2] Δημιουργία dataset για economic signal...")
    X_econ_clean, y_econ_clean = TimeSeriesGenerator.create_dataset(
        signal_econ_clean, window_size=window_size, stride=8
    )
    X_econ_noisy, _ = TimeSeriesGenerator.create_dataset(
        signal_econ_noisy, window_size=window_size, stride=8
    )
    
    # Normalize
    scaler = StandardScaler()
    X_econ_clean_flat = X_econ_clean.reshape(-1, window_size)
    X_econ_clean_flat = scaler.fit_transform(X_econ_clean_flat)
    X_econ_clean = X_econ_clean_flat.reshape(-1, window_size)
    
    X_econ_noisy_flat = X_econ_noisy.reshape(-1, window_size)
    X_econ_noisy_flat = scaler.transform(X_econ_noisy_flat)
    X_econ_noisy = X_econ_noisy_flat.reshape(-1, window_size)
    
    n_train_econ = int(0.6 * len(X_econ_clean))
    n_val_econ = int(0.2 * len(X_econ_clean))
    
    X_econ_train_clean = X_econ_clean[:n_train_econ]
    X_econ_train_noisy = X_econ_noisy[:n_train_econ]
    X_econ_val_clean = X_econ_clean[n_train_econ:n_train_econ+n_val_econ]
    X_econ_val_noisy = X_econ_noisy[n_train_econ:n_train_econ+n_val_econ]
    X_econ_test_clean = X_econ_clean[n_train_econ+n_val_econ:]
    X_econ_test_noisy = X_econ_noisy[n_train_econ+n_val_econ:]
    
    print(f"✓ Train: {len(X_econ_train_clean)}, Val: {len(X_econ_val_clean)}, Test: {len(X_econ_test_clean)}")
    
    # Create PyTorch DataLoaders
    train_ds_econ = DenoisingDataset(X_econ_train_noisy, X_econ_train_clean)
    val_ds_econ = DenoisingDataset(X_econ_val_noisy, X_econ_val_clean)
    test_ds_econ = DenoisingDataset(X_econ_test_noisy, X_econ_test_clean)
    
    train_loader_econ = DataLoader(train_ds_econ, batch_size=batch_size, shuffle=True)
    val_loader_econ = DataLoader(val_ds_econ, batch_size=batch_size, shuffle=False)
    test_loader_econ = DataLoader(test_ds_econ, batch_size=batch_size, shuffle=False)
    
    # ========== 3. ΕΚΠΑΙΔΕΥΣΗ ΜΟΝΤΕΛΩΝ ==========
    print("\n" + "="*80)
    print("ΦΑΣΗ 3: ΕΚΠΑΙΔΕΥΣΗ ΝΕΥΡΩΝΙΚΩΝ ΔΙΚΤΥΩΝ (PyTorch)")
    print("="*80)
    
    # Gravitational Wave: CNN Autoencoder
    print("\n[3.1] GRAVITATIONAL WAVE - CNN Autoencoder")
    model_gw_cnn = Conv1DAutoencoder(input_length=window_size, latent_dim=16)
    train_denoising_model(
        model_gw_cnn, train_loader_gw, val_loader_gw,
        epochs=50, model_name="CNN-Autoencoder (GW)", device=device
    )
    
    # Gravitational Wave: TDNN
    print("\n[3.2] GRAVITATIONAL WAVE - TDNN")
    model_gw_tdnn = TDNN(input_length=window_size, time_delays=(1, 2, 4, 8))
    train_denoising_model(
        model_gw_tdnn, train_loader_gw, val_loader_gw,
        epochs=50, model_name="TDNN (GW)", device=device
    )
    
    # Economic: CNN Autoencoder
    print("\n[3.3] ECONOMIC SIGNAL - CNN Autoencoder")
    model_econ_cnn = Conv1DAutoencoder(input_length=window_size, latent_dim=16)
    train_denoising_model(
        model_econ_cnn, train_loader_econ, val_loader_econ,
        epochs=100, model_name="CNN-Autoencoder (Economic)", device=device
    )
    
    # Economic: TDNN
    print("\n[3.4] ECONOMIC SIGNAL - TDNN")
    model_econ_tdnn = TDNN(input_length=window_size, time_delays=(1, 2, 4, 8))
    train_denoising_model(
        model_econ_tdnn, train_loader_econ, val_loader_econ,
        epochs=100, model_name="TDNN (Economic)", device=device
    )
    
    # ========== 4. ΑΞΙΟΛΟΓΗΣΗ ΜΟΝΤΕΛΩΝ ==========
    print("\n" + "="*80)
    print("ΦΑΣΗ 4: ΑΞΙΟΛΟΓΗΣΗ ΜΟΝΤΕΛΩΝ")
    print("="*80)
    
    results_gw_cnn = evaluate_denoiser(model_gw_cnn, test_loader_gw, "CNN-Autoencoder (GW)", device=device)
    results_gw_tdnn = evaluate_denoiser(model_gw_tdnn, test_loader_gw, "TDNN (GW)", device=device)
    results_econ_cnn = evaluate_denoiser(model_econ_cnn, test_loader_econ, "CNN-Autoencoder (Economic)", device=device)
    results_econ_tdnn = evaluate_denoiser(model_econ_tdnn, test_loader_econ, "TDNN (Economic)", device=device)
    
    # ========== 5. FULL SIGNAL PREDICTIONS ==========
    print("\n" + "="*80)
    print("ΦΑΣΗ 5: PREDICTIONS ΣΕ ΟΛΟΚΛΗΡΟ ΤΟ ΣΗΜΑ")
    print("="*80)
    
    print("\n[5.1] Prediction για Gravitational Wave...")
    gw_pred_cnn = predict_full_signal(model_gw_cnn, signal_gw_noisy, window_size, stride=8, device=device)
    gw_pred_tdnn = predict_full_signal(model_gw_tdnn, signal_gw_noisy, window_size, stride=8, device=device)
    print("✓ Completed")
    
    print("\n[5.2] Prediction για Economic Signal...")
    econ_pred_cnn = predict_full_signal(model_econ_cnn, signal_econ_noisy, window_size, stride=8, device=device)
    econ_pred_tdnn = predict_full_signal(model_econ_tdnn, signal_econ_noisy, window_size, stride=8, device=device)
    print("✓ Completed")
    
    # ========== 6. EVENT DETECTION ==========
    print("\n" + "="*80)
    print("ΦΑΣΗ 6: ΑΝΙΧΝΕΥΣΗ ΣΥΜΒΑΝΤΩΝ (EVENT DETECTION)")
    print("="*80)
    
    print("\n[6.1] GRAVITATIONAL WAVE - Event Detection")
    events_gw = detect_events(signal_gw_clean, signal_gw_noisy, gw_pred_cnn, threshold_std=1.5)
    
    print(f"\nClean signal events: {len(events_gw['clean'])}")
    for i, event in enumerate(events_gw['clean'][:3]):
        print(f"  Event {i+1}: center={event['center']:.1f}, "
              f"amplitude={event['amplitude']:.4f}, max_acc={event['max_acceleration']:.4f}")
    
    print(f"\nNoisy signal events: {len(events_gw['noisy'])}")
    print(f"Denoised (CNN) signal events: {len(events_gw['denoised'])}")
    
    if len(events_gw['clean']) > 0:
        detected = len(events_gw['denoised'])
        true_events = len(events_gw['clean'])
        recall = min(detected / true_events, 1.0)
        print(f"\nEvent Detection Recall: {recall*100:.1f}%")
    
    # Economic events
    print("\n[6.2] ECONOMIC SIGNAL - Crisis Detection")
    events_econ = detect_events(signal_econ_clean, signal_econ_noisy, econ_pred_cnn, threshold_std=1.0)
    
    print(f"Clean signal crises: {len(events_econ['clean'])}")
    print(f"Noisy signal crises: {len(events_econ['noisy'])}")
    print(f"Denoised signal crises: {len(events_econ['denoised'])}")
    
    # ========== 7. ΑΝΑΛΥΣΗ - ΣΥΜΠΕΡΑΣΜΑΤΑ ==========
    print("\n" + "="*80)
    print("ΦΑΣΗ 7: ΑΝΑΛΥΣΗ - ΣΥΜΠΕΡΑΣΜΑΤΑ")
    print("="*80)
    
    print("\n[GRAVITATIONAL WAVES]")
    print(f"✓ CNN Autoencoder: SNR improvement = {results_gw_cnn['snr_improvement']:.2f} dB")
    print(f"✓ TDNN: SNR improvement = {results_gw_tdnn['snr_improvement']:.2f} dB")
    best_gw = 'CNN' if results_gw_cnn['snr_improvement'] > results_gw_tdnn['snr_improvement'] else 'TDNN'
    print(f"➜ Καλύτερο μοντέλο: {best_gw}")
    if len(events_gw['clean']) > 0:
        print(f"  Event detection success: {len(events_gw['denoised']) / len(events_gw['clean']) * 100:.1f}%")
    
    print("\n[ECONOMIC SIGNALS]")
    print(f"✓ CNN Autoencoder: SNR improvement = {results_econ_cnn['snr_improvement']:.2f} dB")
    print(f"✓ TDNN: SNR improvement = {results_econ_tdnn['snr_improvement']:.2f} dB")
    best_econ = 'CNN' if results_econ_cnn['snr_improvement'] > results_econ_tdnn['snr_improvement'] else 'TDNN'
    print(f"➜ Καλύτερο μοντέλο: {best_econ}")
    if len(events_econ['clean']) > 0:
        print(f"  Crisis detection success: {len(events_econ['denoised']) / len(events_econ['clean']) * 100:.1f}%")
    
    # ========== 8. ΕΞΟΔΟΣ ΔΕΔΟΜΕΝΩΝ ==========
    print("\n" + "="*80)
    print("ΦΑΣΗ 8: ΕΞΟΔΟΣ ΑΠΟΤΕΛΕΣΜΑΤΩΝ")
    print("="*80)
    
    # Αποθήκευση
    np.savez('denoising_results_pytorch.npz', 
             signal_gw_clean=signal_gw_clean,
             signal_gw_noisy=signal_gw_noisy,
             signal_gw_denoised=gw_pred_cnn,
             signal_econ_clean=signal_econ_clean,
             signal_econ_noisy=signal_econ_noisy,
             signal_econ_denoised=econ_pred_cnn)
    
    print("✓ Αποτελέσματα αποθηκεύτηκαν στο denoising_results_pytorch.npz")
    
    return {
        'gw_clean': signal_gw_clean,
        'gw_noisy': signal_gw_noisy,
        'gw_denoised': gw_pred_cnn,
        'econ_clean': signal_econ_clean,
        'econ_noisy': signal_econ_noisy,
        'econ_denoised': econ_pred_cnn,
        'models': {
            'gw_cnn': model_gw_cnn,
            'gw_tdnn': model_gw_tdnn,
            'econ_cnn': model_econ_cnn,
            'econ_tdnn': model_econ_tdnn
        }
    }
 
 
if __name__ == "__main__":
    data = main()
    print("\n" + "="*80)
    print("ΟΛΟΚΛΗΡΩΣΗ ΑΝΑΛΥΣΗΣ (PYTORCH)")
    print("="*80)


ΦΑΣΗ 1: ΔΗΜΙΟΥΡΓΙΑ ΣΥΝΘΕΤΙΚΩΝ ΔΕΔΟΜΕΝΩΝ

[1.1] Δημιουργία σήματος βαρυτικών κυμάτων...
✓ Signal length: 8000 samples
  Clean signal power: 0.182815
  Noise power: 0.058447

[1.2] Δημιουργία οικονομικού σήματος (S&P 500)...
✓ Signal length: 2000 samples
  Range: [-1280.75, 9.80]

ΦΑΣΗ 2: ΠΡΟΕΤΟΙΜΑΣΙΑ DATASETS

[2.1] Δημιουργία dataset για GW signal (window_size=64)...
✓ Train: 595, Val: 198, Test: 199

[2.2] Δημιουργία dataset για economic signal...
✓ Train: 145, Val: 48, Test: 49

ΦΑΣΗ 3: ΕΚΠΑΙΔΕΥΣΗ ΝΕΥΡΩΝΙΚΩΝ ΔΙΚΤΥΩΝ (PyTorch)

[3.1] GRAVITATIONAL WAVE - CNN Autoencoder

ΕΚΠΑΙΔΕΥΣΗ ΜΟΝΤΕΛΟΥ: CNN-Autoencoder (GW)
Epoch 1/50 - Train Loss: 0.109829, Val Loss: 0.166224
Epoch 5/50 - Train Loss: 0.009958, Val Loss: 0.081295
Epoch 10/50 - Train Loss: 0.005792, Val Loss: 0.072520
Epoch 15/50 - Train Loss: 0.004128, Val Loss: 0.073718
Epoch 20/50 - Train Loss: 0.003420, Val Loss: 0.069532
Epoch 25/50 - Train Loss: 0.003625, Val Loss: 0.066503
Epoch 30/50 - Train Loss: 0.002428, Val Loss: 0.06

In [ ]:
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": 36,
   "id": "4cb7e72f",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Using device: cuda\n"
     ]
    }
   ],
   "source": [
    "'''\n",
    "Ανάλυση χρονοσειρων με Denoising μέσω PyTorch 1D Convolutional Neural Network (CNNs) και Time Delay Neural Networks TDNNs\n",
    " \n",
    "'''\n",
    "# Εισαγωγή των απαραίτητων βιβλιοθηκών\n",
    "\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt \n",
    "import seaborn as sns \n",
    "from scipy import signal \n",
    "from scipy.stats import norm \n",
    "import pandas as pd \n",
    "from sklearn.preprocessing import StandardScaler\n",
    "from sklearn.metrics import mean_squared_error, mean_absolute_error\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "import torch \n",
    "import torch.nn as nn \n",
    "import torch.optim as optim \n",
    "from torch.utils.data import Dataset, DataLoader \n",
    "import torch.nn.functional as F\n",
    "import yfinance as yf \n",
    "\n",
    "\n",
    "# Device setup \n",
    "device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n",
    "print(f'Using device: {device}')\n",
    "\n",
    "#Seed για αναπαραγωγιμότητα\n",
    "np.random.seed(42)\n",
    "torch.manual_seed(42)\n",
    "if torch.cuda.is_available():\n",
    "    torch.cuda.manual_seed_all(42)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "7540372c",
   "metadata": {},
   "outputs": [],
   "source": []
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "5efa6c8a",
   "metadata": {},
   "outputs": [],
   "source": [
    "class TimeSeriesGenerator: \n",
    "    @staticmethod\n",
    "    def gravitational_wave_signal(duration=2.0, sampling_rate=5000, frequency_start=20,frequency_end=250):\n",
    "        t = np.linspace(0, duration, int(duration * sampling_rate), endpoint=False)\n",
    "        chirp = signal.chirp(t, frequency_start, duration, frequency_end, method='linear')\n",
    "        envelope = np.exp(-((t - duration/2)**2) / (2 * (duration/5)**2))\n",
    "        signal_clean = chirp * envelope\n",
    "        \n",
    "        n_spikes = 10\n",
    "        spike_positions = np.random.random_integers(0, len(signal_clean), n_spikes)\n",
    "        spike_amplitudes = np.random.uniform(0.5, 2, n_spikes)\n",
    "        for pos, amp in zip(spike_positions, spike_amplitudes):\n",
    "            signal_clean[max(0, pos-5):min(len(signal_clean), pos+5)] += amp * signal.windows.hann(10)\n",
    "        return t, signal_clean\n",
    "    @staticmethod\n",
    "    def economic_signal(length=1000, volatility=0.02):\n",
    "        \"\"\"\n",
    "        Σήμα οικονομικών δεδομένων (log-returns)\n",
    "        Με περιόδους κρίσης (regime shifts)\n",
    "        \"\"\"\n",
    "        # Drift\n",
    "        drift = 0.0001\n",
    "        \n",
    "        # Δημιουργία log-returns με GBM\n",
    "        returns = np.zeros(length)\n",
    "        returns[0] = 0\n",
    "        \n",
    "        # Regime shifts (κρίσεις)\n",
    "        regime = np.ones(length)\n",
    "        crisis_periods = [\n",
    "            (100, 200, 3.0),   # Κρίση 1: χρόνος, διάρκεια, σοβαρότητα\n",
    "            (400, 150, 2.5),   # Κρίση 2\n",
    "            (700, 120, 2.0)    # Κρίση 3\n",
    "        ]\n",
    "        \n",
    "        for start, duration, severity in crisis_periods:\n",
    "            regime[start:start+duration] = severity\n",
    "        \n",
    "        for i in range(1, length):\n",
    "            shock = np.random.normal(drift, volatility * regime[i])\n",
    "            returns[i] = returns[i-1] + shock\n",
    "        \n",
    "        # Cumulative sum για τιμές\n",
    "        prices = np.cumsum(returns)\n",
    "        \n",
    "        return np.arange(length), prices\n",
    "    @staticmethod\n",
    "    def add_noise(signal_clean, snr_db=10):\n",
    "        \"\"\"\n",
    "        Add Gaussian noise with a specific SNR (Signal-to-Noise Ratio) in dB.\n",
    "        SNR (dB) = 10 * log10(P_signal / P_noise)\n",
    "        \"\"\"\n",
    "        signal_power = np.mean(signal_clean ** 2)\n",
    "        noise_power = signal_power / (10 ** (snr_db / 10))\n",
    "        noise = np.random.normal(0, np.sqrt(noise_power), len(signal_clean))\n",
    "        signal_noisy = signal_clean + noise\n",
    "        return signal_noisy\n",
    "    @staticmethod\n",
    "    def create_dataset(signal_clean, window_size=64, stride=8):\n",
    "        X, y= [], []\n",
    "        \n",
    "        for i in range(0, len(signal_clean) - window_size, stride):\n",
    "            X.append(signal_clean[i:i+window_size])\n",
    "            y.append(signal_clean[i+window_size])\n",
    "        return np.array(X), np.array(y)\n",
    "\n",
    "class Conv1DAutoencoder(nn.Module):\n",
    "    \"\"\"\n",
    "    1D Convolutional Autoencoder για denoising\n",
    "    \"\"\"\n",
    "    \n",
    "    def __init__(self, input_length=64, latent_dim=16):\n",
    "        super(Conv1DAutoencoder, self).__init__()\n",
    "        \n",
    "        self.input_length = input_length\n",
    "        self.latent_dim = latent_dim\n",
    "        \n",
    "        # Encoder\n",
    "        self.enc_conv1 = nn.Conv1d(1, 32, kernel_size=5, padding=2, stride=1)\n",
    "        self.enc_bn1 = nn.BatchNorm1d(32)\n",
    "        self.enc_pool1 = nn.MaxPool1d(kernel_size=2)\n",
    "        \n",
    "        self.enc_conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2, stride=1)\n",
    "        self.enc_bn2 = nn.BatchNorm1d(64)\n",
    "        self.enc_pool2 = nn.MaxPool1d(kernel_size=2)\n",
    "        \n",
    "        self.enc_conv3 = nn.Conv1d(64, 128, kernel_size=3, padding=1, stride=1)\n",
    "        self.enc_bn3 = nn.BatchNorm1d(128)\n",
    "        \n",
    "        # Bottleneck\n",
    "        bottleneck_size = 128 * (input_length // 4)\n",
    "        self.bottleneck_fc1 = nn.Linear(bottleneck_size, latent_dim)\n",
    "        self.bottleneck_fc2 = nn.Linear(latent_dim, bottleneck_size)\n",
    "        \n",
    "        # Decoder\n",
    "        self.dec_conv1 = nn.Conv1d(128, 128, kernel_size=3, padding=1)\n",
    "        self.dec_bn1 = nn.BatchNorm1d(128)\n",
    "        self.dec_up1 = nn.Upsample(scale_factor=2, mode='nearest')\n",
    "        \n",
    "        self.dec_conv2 = nn.Conv1d(128, 64, kernel_size=5, padding=2)\n",
    "        self.dec_bn2 = nn.BatchNorm1d(64)\n",
    "        self.dec_up2 = nn.Upsample(scale_factor=2, mode='nearest')\n",
    "        \n",
    "        self.dec_conv3 = nn.Conv1d(64, 32, kernel_size=5, padding=2)\n",
    "        self.dec_bn3 = nn.BatchNorm1d(32)\n",
    "        \n",
    "        self.dec_conv4 = nn.Conv1d(32, 1, kernel_size=5, padding=2)\n",
    "    \n",
    "    def encode(self, x):\n",
    "        # x: (batch, 1, window_size)\n",
    "        x = F.relu(self.enc_bn1(self.enc_conv1(x)))\n",
    "        x = self.enc_pool1(x)\n",
    "        \n",
    "        x = F.relu(self.enc_bn2(self.enc_conv2(x)))\n",
    "        x = self.enc_pool2(x)\n",
    "        \n",
    "        x = F.relu(self.enc_bn3(self.enc_conv3(x)))\n",
    "        \n",
    "        x = x.view(x.size(0), -1)\n",
    "        x = F.relu(self.bottleneck_fc1(x))\n",
    "        \n",
    "        return x\n",
    "    \n",
    "    def decode(self, z):\n",
    "        # z: (batch, latent_dim)\n",
    "        x = F.relu(self.bottleneck_fc2(z))\n",
    "        x = x.view(x.size(0), 128, -1)\n",
    "        \n",
    "        x = F.relu(self.dec_bn1(self.dec_conv1(x)))\n",
    "        x = self.dec_up1(x)\n",
    "        \n",
    "        x = F.relu(self.dec_bn2(self.dec_conv2(x)))\n",
    "        x = self.dec_up2(x)\n",
    "        \n",
    "        x = F.relu(self.dec_bn3(self.dec_conv3(x)))\n",
    "        x = self.dec_conv4(x)\n",
    "        \n",
    "        return x\n",
    "    \n",
    "    def forward(self, x):\n",
    "        z = self.encode(x)\n",
    "        x_recon = self.decode(z)\n",
    "        return x_recon\n",
    "    \n",
    "class TDNN(nn.Module):\n",
    "    \"\"\"\n",
    "    Time Delay Neural Network (TDNN)\n",
    "    Χρησιμοποιεί temporal context windows για feature extraction\n",
    "    \"\"\"\n",
    "    \n",
    "    def __init__(self, input_length=64, time_delays=(1, 2, 4, 8)):\n",
    "        super(TDNN, self).__init__()\n",
    "        \n",
    "        self.input_length = input_length\n",
    "        self.time_delays = time_delays\n",
    "        \n",
    "        # Number of delayed features = 1 (original) + len(time_delays)\n",
    "        n_delayed_features = 1 + len(time_delays)\n",
    "        \n",
    "        # Dense layers\n",
    "        self.fc1 = nn.Linear(input_length * n_delayed_features, 128)\n",
    "        self.bn1 = nn.BatchNorm1d(128)\n",
    "        self.dropout1 = nn.Dropout(0.2)\n",
    "        \n",
    "        self.fc2 = nn.Linear(128, 64)\n",
    "        self.bn2 = nn.BatchNorm1d(64)\n",
    "        self.dropout2 = nn.Dropout(0.2)\n",
    "        \n",
    "        self.fc3 = nn.Linear(64, 32)\n",
    "        self.fc4 = nn.Linear(32, input_length)\n",
    "    \n",
    "    def forward(self, x):\n",
    "        # x: (batch, 1, window_size)\n",
    "        batch_size = x.size(0)\n",
    "        x = x.squeeze(1)  # (batch, window_size)\n",
    "        \n",
    "        # Create time-delayed features\n",
    "        delayed_features = [x]  # Original signal\n",
    "        \n",
    "        for delay in self.time_delays:\n",
    "            if delay > 0:\n",
    "                # Shift signal by delay and pad with zeros\n",
    "                delayed = torch.cat([\n",
    "                    torch.zeros(batch_size, delay, device=x.device),\n",
    "                    x[:, :-delay]\n",
    "                ], dim=1)\n",
    "                delayed_features.append(delayed)\n",
    "        \n",
    "        # Concatenate all time-delayed versions\n",
    "        x_delayed = torch.cat(delayed_features, dim=1)  # (batch, window_size * n_delayed)\n",
    "        \n",
    "        # Dense layers\n",
    "        x = F.relu(self.bn1(self.fc1(x_delayed)))\n",
    "        x = self.dropout1(x)\n",
    "        \n",
    "        x = F.relu(self.bn2(self.fc2(x)))\n",
    "        x = self.dropout2(x)\n",
    "        \n",
    "        x = F.relu(self.fc3(x))\n",
    "        x = self.fc4(x)\n",
    "        \n",
    "        # Reshape to (batch, 1, window_size) for compatibility\n",
    "        x = x.unsqueeze(1)\n",
    "        \n",
    "        return x\n",
    "class DenoisingDataset(Dataset):\n",
    "    \"\"\"PyTorch Dataset για denoising\"\"\"\n",
    "    \n",
    "    def __init__(self, X_noisy, y_clean):\n",
    "        \"\"\"\n",
    "        Args:\n",
    "            X_noisy: (N, window_size) noisy signals\n",
    "            y_clean: (N, window_size) clean signals\n",
    "        \"\"\"\n",
    "        self.X_noisy = torch.FloatTensor(X_noisy)\n",
    "        self.y_clean = torch.FloatTensor(y_clean)\n",
    "    \n",
    "    def __len__(self):\n",
    "        return len(self.X_noisy)\n",
    "    \n",
    "    def __getitem__(self, idx):\n",
    "        # Reshape to (1, window_size) for 1D convolution\n",
    "        x = self.X_noisy[idx].unsqueeze(0)\n",
    "        y = self.y_clean[idx].unsqueeze(0)\n",
    "        return x, y"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 38,
   "id": "44101d36",
   "metadata": {},
   "outputs": [],
   "source": [
    "def train_denoising_model(model, train_loader, val_loader, epochs=30, \n",
    "                         model_name=\"Model\", device=device):\n",
    "    \"\"\"Εκπαίδευση μοντέλου denoising με PyTorch\"\"\"\n",
    "    \n",
    "    print(f\"\\n{'='*80}\")\n",
    "    print(f\"ΕΚΠΑΙΔΕΥΣΗ ΜΟΝΤΕΛΟΥ: {model_name}\")\n",
    "    print(f\"{'='*80}\")\n",
    "    model = model.to(device)\n",
    "    optimizer = optim.Adam(model.parameters(), lr = 0.001)\n",
    "    criterion = nn.MSELoss()\n",
    "    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)\n",
    "    best_val_loss = float('inf')\n",
    "    patience = 10\n",
    "    patience_counter = 0\n",
    "    train_losses, val_losses = [], []\n",
    "    for epoch in range(epochs):\n",
    "        model.train()\n",
    "        train_loss = 0.0\n",
    "        for batch_noisy, batch_clean in train_loader:\n",
    "                batch_noisy, batch_clean = batch_noisy.to(device), batch_clean.to(device)\n",
    "                optimizer.zero_grad()\n",
    "                outputs = model(batch_noisy)\n",
    "                loss = criterion(outputs, batch_clean)\n",
    "                loss.backward()\n",
    "                optimizer.step()\n",
    "                train_loss += loss.item() \n",
    "        train_loss /= len(train_loader)\n",
    "        train_losses.append(train_loss)\n",
    "        model.eval()\n",
    "        val_loss = 0.0\n",
    "        with torch.no_grad():\n",
    "            for batch_noisy, batch_clean in val_loader:\n",
    "                batch_noisy, batch_clean = batch_noisy.to(device), batch_clean.to(device)\n",
    "                outputs = model(batch_noisy)\n",
    "                loss = criterion(outputs, batch_clean)\n",
    "                val_loss += loss.item()\n",
    "        val_loss /= len(val_loader)\n",
    "        val_losses.append(val_loss)\n",
    "        scheduler.step(val_loss)\n",
    "        if val_loss < best_val_loss:\n",
    "            best_val_loss = val_loss\n",
    "            patience_counter = 0\n",
    "        else:\n",
    "            patience_counter += 1\n",
    "        \n",
    "        if patience_counter >= patience:\n",
    "            print(f\"Early stopping at epoch {epoch+1}\")\n",
    "            break\n",
    "        if (epoch + 1) % 5 == 0 or epoch == 0:\n",
    "            print(f\"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}\")\n",
    "    \n",
    "    print(f\"✓ Εκπαίδευση ολοκληρώθηκε\")\n",
    "    print(f\"  Final training loss: {train_losses[-1]:.6f}\")\n",
    "    print(f\"  Final validation loss: {val_losses[-1]:.6f}\")\n",
    "\n",
    "    return train_losses, val_losses"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 39,
   "id": "d1ca4479",
   "metadata": {},
   "outputs": [],
   "source": [
    "def evaluate_denoiser(model, test_loader, model_name=\"model\", device=device):\n",
    "    \"\"\"Αξιολόγηση μοντέλου\"\"\"\n",
    "    \n",
    "    print(f\"\\n{'='*60}\")\n",
    "    print(f\"ΑΞΙΟΛΟΓΗΣΗ: {model_name}\")\n",
    "    print(f\"{'='*60}\")\n",
    "    \n",
    "    model.eval()\n",
    "    \n",
    "    all_noisy = []\n",
    "    all_clean = []\n",
    "    all_denoised = []\n",
    "    \n",
    "    with torch.no_grad():\n",
    "        for batch_noisy, batch_clean in test_loader:\n",
    "            batch_noisy, batch_clean = batch_noisy.to(device), batch_clean.to(device)\n",
    "            outputs = model(batch_noisy)\n",
    "            all_noisy.append(batch_noisy.cpu().numpy())\n",
    "            all_clean.append(batch_clean.cpu().numpy())\n",
    "            all_denoised.append(outputs.cpu().numpy())\n",
    "    X_test_noisy = np.concatenate(all_noisy, axis=0)\n",
    "    X_test_clean = np.concatenate(all_clean, axis=0)\n",
    "    X_test_denoised = np.concatenate(all_denoised, axis=0)\n",
    "    \n",
    "    # Metrics\n",
    "    mse_noisy = np.mean((X_test_noisy - X_test_clean) ** 2)\n",
    "    mse_denoised = np.mean((X_test_denoised - X_test_clean) ** 2)\n",
    "    mae_noisy = np.mean(np.abs(X_test_noisy - X_test_clean))\n",
    "    mae_denoised = np.mean(np.abs(X_test_denoised - X_test_clean))\n",
    "    \n",
    "    # SNR improvement\n",
    "    snr_noise = 10 * np.log10(np.mean(X_test_clean ** 2) / mse_noisy + 1e-10)\n",
    "    snr_denoised = 10 * np.log10(np.mean(X_test_clean ** 2) / mse_denoised + 1e-10)\n",
    "    snr_improvement = snr_denoised - snr_noise\n",
    "    \n",
    "    print(f\"MSE (noisy):     {mse_noisy:.6f}\")\n",
    "    print(f\"MSE (denoised):  {mse_denoised:.6f}\")\n",
    "    print(f\"Βελτίωση MSE:    {(1 - mse_denoised/mse_noisy)*100:.2f}%\")\n",
    "    print(f\"\\nMAE (noisy):     {mae_noisy:.6f}\")\n",
    "    print(f\"MAE (denoised):  {mae_denoised:.6f}\")\n",
    "    print(f\"Βελτίωση MAE:    {(1 - mae_denoised/mae_noisy)*100:.2f}%\")\n",
    "    print(f\"\\nSNR (noisy):     {snr_noise:.2f} dB\")\n",
    "    print(f\"SNR (denoised):  {snr_denoised:.2f} dB\")\n",
    "    print(f\"SNR Improvement: {snr_improvement:.2f} dB\")\n",
    "    \n",
    "    return {\n",
    "        'mse_noisy': mse_noisy,\n",
    "        'mse_denoised': mse_denoised,\n",
    "        'mae_noisy': mae_noisy,\n",
    "        'mae_denoised': mae_denoised,\n",
    "        'snr_noisy': snr_noise,\n",
    "        'snr_denoised': snr_denoised,\n",
    "        'snr_improvement': snr_improvement,\n",
    "        'predictions': X_test_denoised\n",
    "    }\n",
    "    \n",
    "def predict_full_signal(model, signal_noisy, window_size=64, stride=8, device=device):\n",
    "    \"\"\"Prediction σε ολόκληρο το σήμα\"\"\"\n",
    "    model.eval()\n",
    "    predictions = np.zeros_like(signal_noisy)\n",
    "    counts = np.zeros_like(signal_noisy)\n",
    "    \n",
    "    with torch.no_grad():\n",
    "        for i in range(0, len(signal_noisy) - window_size, stride):\n",
    "            window = signal_noisy[i:i + window_size]\n",
    "            window_tensor = torch.FloatTensor(window).unsqueeze(0).unsqueeze(0).to(device)\n",
    "            \n",
    "            output = model(window_tensor)\n",
    "            output_np = output.squeeze().cpu().numpy()\n",
    "            \n",
    "            predictions[i:i + window_size] += output_np\n",
    "            counts[i:i + window_size] += 1\n",
    "    \n",
    "    # Average overlapping regions\n",
    "    predictions = np.divide(predictions, counts, where=counts > 0, out=predictions)\n",
    "    \n",
    "    return predictions"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 40,
   "id": "96873edd",
   "metadata": {},
   "outputs": [],
   "source": [
    "def detect_events(signal_clean, signal_noisy, signal_denoised, threshold_std=2.0):\n",
    "    \"\"\"Ανίχνευση σημαντικών συμβάντων (events)\"\"\"\n",
    "    \n",
    "    results = {\n",
    "        'clean': [],\n",
    "        'noisy': [],\n",
    "        'denoised': []\n",
    "    }\n",
    "    \n",
    "    signal_clean = np.squeeze(np.array(signal_clean))\n",
    "    signal_noisy = np.squeeze(np.array(signal_noisy))\n",
    "    signal_denoised = np.squeeze(np.array(signal_denoised))\n",
    "    \n",
    "    for name, sig in [('clean', signal_clean), ('noisy', signal_noisy), \n",
    "                      ('denoised', signal_denoised)]:\n",
    "        \n",
    "        velocity = np.gradient(sig)\n",
    "        acceleration = np.gradient(velocity)\n",
    "        \n",
    "        threshold = threshold_std * np.std(acceleration)\n",
    "        event_mask = np.abs(acceleration) > threshold\n",
    "        event_indices = np.where(event_mask)[0]\n",
    "        \n",
    "        event_clusters = []\n",
    "        if len(event_indices) > 0:\n",
    "            current_cluster = [event_indices[0]]\n",
    "            for idx in event_indices[1:]:\n",
    "                if idx - current_cluster[-1] <= 5:\n",
    "                    current_cluster.append(idx)\n",
    "                else:\n",
    "                    if len(current_cluster) > 0:\n",
    "                        event_clusters.append(current_cluster[:])\n",
    "                    current_cluster = [idx]\n",
    "            if len(current_cluster) > 0:\n",
    "                event_clusters.append(current_cluster[:])\n",
    "        \n",
    "        for cluster in event_clusters:\n",
    "            cluster_arr = np.array(cluster, dtype=int)\n",
    "            try:\n",
    "                event_data = {\n",
    "                    'center': float(np.mean(cluster_arr)),\n",
    "                    'duration': int(len(cluster_arr)),\n",
    "                    'max_acceleration': float(np.max(np.abs(acceleration[cluster_arr]))),\n",
    "                    'max_velocity': float(np.max(np.abs(velocity[cluster_arr]))),\n",
    "                    'amplitude': float(np.max(sig[cluster_arr]) - np.min(sig[cluster_arr]))\n",
    "                }\n",
    "                results[name].append(event_data)\n",
    "            except:\n",
    "                pass\n",
    "    \n",
    "    return results"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "718fe225",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "================================================================================\n",
      "ΦΑΣΗ 1: ΔΗΜΙΟΥΡΓΙΑ ΣΥΝΘΕΤΙΚΩΝ ΔΕΔΟΜΕΝΩΝ\n",
      "================================================================================\n",
      "\n",
      "[1.1] Δημιουργία σήματος βαρυτικών κυμάτων...\n",
      "✓ Signal length: 8000 samples\n",
      "  Clean signal power: 0.182815\n",
      "  Noise power: 0.058447\n",
      "\n",
      "[1.2] Δημιουργία οικονομικού σήματος (S&P 500)...\n",
      "✓ Signal length: 2000 samples\n",
      "  Range: [-1280.75, 9.80]\n",
      "\n",
      "================================================================================\n",
      "ΦΑΣΗ 2: ΠΡΟΕΤΟΙΜΑΣΙΑ DATASETS\n",
      "================================================================================\n",
      "\n",
      "[2.1] Δημιουργία dataset για GW signal (window_size=64)...\n",
      "✓ Train: 595, Val: 198, Test: 199\n",
      "\n",
      "[2.2] Δημιουργία dataset για economic signal...\n",
      "✓ Train: 145, Val: 48, Test: 49\n",
      "\n",
      "================================================================================\n",
      "ΦΑΣΗ 3: ΕΚΠΑΙΔΕΥΣΗ ΝΕΥΡΩΝΙΚΩΝ ΔΙΚΤΥΩΝ (PyTorch)\n",
      "================================================================================\n",
      "\n",
      "[3.1] GRAVITATIONAL WAVE - CNN Autoencoder\n",
      "\n",
      "================================================================================\n",
      "ΕΚΠΑΙΔΕΥΣΗ ΜΟΝΤΕΛΟΥ: CNN-Autoencoder (GW)\n",
      "================================================================================\n",
      "Epoch 1/50 - Train Loss: 0.109829, Val Loss: 0.166224\n",
      "Epoch 5/50 - Train Loss: 0.009958, Val Loss: 0.081295\n",
      "Epoch 10/50 - Train Loss: 0.005792, Val Loss: 0.072520\n",
      "Epoch 15/50 - Train Loss: 0.004128, Val Loss: 0.073718\n",
      "Epoch 20/50 - Train Loss: 0.003420, Val Loss: 0.069532\n",
      "Epoch 25/50 - Train Loss: 0.003625, Val Loss: 0.066503\n",
      "Epoch 30/50 - Train Loss: 0.002428, Val Loss: 0.067649\n",
      "Epoch 35/50 - Train Loss: 0.001530, Val Loss: 0.067644\n",
      "Epoch 40/50 - Train Loss: 0.001480, Val Loss: 0.065812\n",
      "Epoch 45/50 - Train Loss: 0.000891, Val Loss: 0.065728\n",
      "Early stopping at epoch 46\n",
      "✓ Εκπαίδευση ολοκληρώθηκε\n",
      "  Final training loss: 0.001096\n",
      "  Final validation loss: 0.067269\n",
      "\n",
      "[3.2] GRAVITATIONAL WAVE - TDNN\n",
      "\n",
      "================================================================================\n",
      "ΕΚΠΑΙΔΕΥΣΗ ΜΟΝΤΕΛΟΥ: TDNN (GW)\n",
      "================================================================================\n",
      "Epoch 1/50 - Train Loss: 0.223935, Val Loss: 0.174513\n",
      "Epoch 5/50 - Train Loss: 0.063504, Val Loss: 0.114606\n",
      "Epoch 10/50 - Train Loss: 0.038004, Val Loss: 0.096469\n",
      "Epoch 15/50 - Train Loss: 0.035139, Val Loss: 0.092226\n",
      "Epoch 20/50 - Train Loss: 0.031863, Val Loss: 0.087199\n",
      "Epoch 25/50 - Train Loss: 0.029553, Val Loss: 0.084376\n",
      "Epoch 30/50 - Train Loss: 0.026995, Val Loss: 0.082851\n",
      "Epoch 35/50 - Train Loss: 0.024287, Val Loss: 0.082659\n",
      "Epoch 40/50 - Train Loss: 0.023636, Val Loss: 0.079160\n",
      "Epoch 45/50 - Train Loss: 0.021935, Val Loss: 0.076712\n",
      "Epoch 50/50 - Train Loss: 0.022621, Val Loss: 0.075232\n",
      "✓ Εκπαίδευση ολοκληρώθηκε\n",
      "  Final training loss: 0.022621\n",
      "  Final validation loss: 0.075232\n",
      "\n",
      "[3.3] ECONOMIC SIGNAL - CNN Autoencoder\n",
      "\n",
      "================================================================================\n",
      "ΕΚΠΑΙΔΕΥΣΗ ΜΟΝΤΕΛΟΥ: CNN-Autoencoder (Economic)\n",
      "================================================================================\n",
      "Epoch 1/100 - Train Loss: 0.630122, Val Loss: 0.738315\n",
      "Epoch 5/100 - Train Loss: 0.023683, Val Loss: 0.048606\n",
      "Epoch 10/100 - Train Loss: 0.018873, Val Loss: 0.046574\n",
      "Epoch 15/100 - Train Loss: 0.015847, Val Loss: 0.072070\n",
      "Epoch 20/100 - Train Loss: 0.010489, Val Loss: 0.065126\n",
      "Epoch 25/100 - Train Loss: 0.024772, Val Loss: 0.024908\n",
      "Early stopping at epoch 28\n",
      "✓ Εκπαίδευση ολοκληρώθηκε\n",
      "  Final training loss: 0.015140\n",
      "  Final validation loss: 0.045183\n",
      "\n",
      "[3.4] ECONOMIC SIGNAL - TDNN\n",
      "\n",
      "================================================================================\n",
      "ΕΚΠΑΙΔΕΥΣΗ ΜΟΝΤΕΛΟΥ: TDNN (Economic)\n",
      "================================================================================\n",
      "Epoch 1/100 - Train Loss: 0.883502, Val Loss: 0.751543\n",
      "Epoch 5/100 - Train Loss: 0.385610, Val Loss: 0.794131\n",
      "Epoch 10/100 - Train Loss: 0.113702, Val Loss: 0.769333\n",
      "Early stopping at epoch 11\n",
      "✓ Εκπαίδευση ολοκληρώθηκε\n",
      "  Final training loss: 0.090730\n",
      "  Final validation loss: 0.763548\n",
      "\n",
      "================================================================================\n",
      "ΦΑΣΗ 4: ΑΞΙΟΛΟΓΗΣΗ ΜΟΝΤΕΛΩΝ\n",
      "================================================================================\n",
      "\n",
      "============================================================\n",
      "ΑΞΙΟΛΟΓΗΣΗ: CNN-Autoencoder (GW)\n",
      "============================================================\n",
      "MSE (noisy):     0.058065\n",
      "MSE (denoised):  0.026233\n",
      "Βελτίωση MSE:    54.82%\n",
      "\n",
      "MAE (noisy):     0.191305\n",
      "MAE (denoised):  0.120858\n",
      "Βελτίωση MAE:    36.82%\n",
      "\n",
      "SNR (noisy):     -2.86 dB\n",
      "SNR (denoised):  0.59 dB\n",
      "SNR Improvement: 3.45 dB\n",
      "\n",
      "============================================================\n",
      "ΑΞΙΟΛΟΓΗΣΗ: TDNN (GW)\n",
      "============================================================\n",
      "MSE (noisy):     0.058065\n",
      "MSE (denoised):  0.026491\n",
      "Βελτίωση MSE:    54.38%\n",
      "\n",
      "MAE (noisy):     0.191305\n",
      "MAE (denoised):  0.115737\n",
      "Βελτίωση MAE:    39.50%\n",
      "\n",
      "SNR (noisy):     -2.86 dB\n",
      "SNR (denoised):  0.55 dB\n",
      "SNR Improvement: 3.41 dB\n",
      "\n",
      "============================================================\n",
      "ΑΞΙΟΛΟΓΗΣΗ: CNN-Autoencoder (Economic)\n",
      "============================================================\n",
      "MSE (noisy):     0.091772\n",
      "MSE (denoised):  0.086560\n",
      "Βελτίωση MSE:    5.68%\n",
      "\n",
      "MAE (noisy):     0.244777\n",
      "MAE (denoised):  0.261183\n",
      "Βελτίωση MAE:    -6.70%\n",
      "\n",
      "SNR (noisy):     12.62 dB\n",
      "SNR (denoised):  12.87 dB\n",
      "SNR Improvement: 0.25 dB\n",
      "\n",
      "============================================================\n",
      "ΑΞΙΟΛΟΓΗΣΗ: TDNN (Economic)\n",
      "============================================================\n",
      "MSE (noisy):     0.091772\n",
      "MSE (denoised):  1.701360\n",
      "Βελτίωση MSE:    -1753.91%\n",
      "\n",
      "MAE (noisy):     0.244777\n",
      "MAE (denoised):  1.292753\n",
      "Βελτίωση MAE:    -428.14%\n",
      "\n",
      "SNR (noisy):     12.62 dB\n",
      "SNR (denoised):  -0.06 dB\n",
      "SNR Improvement: -12.68 dB\n",
      "\n",
      "================================================================================\n",
      "ΦΑΣΗ 5: PREDICTIONS ΣΕ ΟΛΟΚΛΗΡΟ ΤΟ ΣΗΜΑ\n",
      "================================================================================\n",
      "\n",
      "[5.1] Prediction για Gravitational Wave...\n",
      "✓ Completed\n",
      "\n",
      "[5.2] Prediction για Economic Signal...\n",
      "✓ Completed\n",
      "\n",
      "================================================================================\n",
      "ΦΑΣΗ 6: ΑΝΙΧΝΕΥΣΗ ΣΥΜΒΑΝΤΩΝ (EVENT DETECTION)\n",
      "================================================================================\n",
      "\n",
      "[6.1] GRAVITATIONAL WAVE - Event Detection\n",
      "\n",
      "Clean signal events: 172\n",
      "  Event 1: center=333.5, amplitude=0.5082, max_acc=0.1003\n",
      "  Event 2: center=765.5, amplitude=1.4161, max_acc=0.2794\n",
      "  Event 3: center=802.5, amplitude=1.1071, max_acc=0.2138\n",
      "\n",
      "Noisy signal events: 417\n",
      "Denoised (CNN) signal events: 173\n",
      "\n",
      "Event Detection Recall: 100.0%\n",
      "\n",
      "[6.2] ECONOMIC SIGNAL - Crisis Detection\n",
      "Clean signal crises: 91\n",
      "Noisy signal crises: 93\n",
      "Denoised signal crises: 60\n",
      "\n",
      "================================================================================\n",
      "ΦΑΣΗ 7: ΑΝΑΛΥΣΗ - ΣΥΜΠΕΡΑΣΜΑΤΑ\n",
      "================================================================================\n",
      "\n",
      "[GRAVITATIONAL WAVES]\n",
      "✓ CNN Autoencoder: SNR improvement = 3.45 dB\n",
      "✓ TDNN: SNR improvement = 3.41 dB\n",
      "➜ Καλύτερο μοντέλο: CNN\n",
      "  Event detection success: 100.6%\n",
      "\n",
      "[ECONOMIC SIGNALS]\n",
      "✓ CNN Autoencoder: SNR improvement = 0.25 dB\n",
      "✓ TDNN: SNR improvement = -12.68 dB\n",
      "➜ Καλύτερο μοντέλο: CNN\n",
      "  Crisis detection success: 65.9%\n",
      "\n",
      "================================================================================\n",
      "ΦΑΣΗ 8: ΕΞΟΔΟΣ ΑΠΟΤΕΛΕΣΜΑΤΩΝ\n",
      "================================================================================\n",
      "✓ Αποτελέσματα αποθηκεύτηκαν στο denoising_results_pytorch.npz\n",
      "\n",
      "================================================================================\n",
      "ΟΛΟΚΛΗΡΩΣΗ ΑΝΑΛΥΣΗΣ (PYTORCH)\n",
      "================================================================================\n"
     ]
    }
   ],
   "source": [
    "def main():\n",
    "    \n",
    "    # ========== 1. ΔΗΜΙΟΥΡΓΙΑ ΔΕΔΟΜΕΝΩΝ ==========\n",
    "    print(\"\\n\" + \"=\"*80)\n",
    "    print(\"ΦΑΣΗ 1: ΔΗΜΙΟΥΡΓΙΑ ΣΥΝΘΕΤΙΚΩΝ ΔΕΔΟΜΕΝΩΝ\")\n",
    "    print(\"=\"*80)\n",
    "    \n",
    "    # Gravitational wave signal\n",
    "    print(\"\\n[1.1] Δημιουργία σήματος βαρυτικών κυμάτων...\")\n",
    "    t_gw, signal_gw_clean = TimeSeriesGenerator.gravitational_wave_signal(\n",
    "        duration=2.0, sampling_rate=4000\n",
    "    )\n",
    "    signal_gw_noisy = TimeSeriesGenerator.add_noise(signal_gw_clean, snr_db=5)\n",
    "    print(f\"✓ Signal length: {len(signal_gw_clean)} samples\")\n",
    "    print(f\"  Clean signal power: {np.mean(signal_gw_clean**2):.6f}\")\n",
    "    print(f\"  Noise power: {np.mean((signal_gw_noisy - signal_gw_clean)**2):.6f}\")\n",
    "    \n",
    "    # Economic signal\n",
    "    print(\"\\n[1.2] Δημιουργία οικονομικού σήματος (S&P 500)...\")\n",
    "    t_econ, signal_econ_clean = TimeSeriesGenerator.economic_signal(length=2000)\n",
    "    signal_econ_noisy = TimeSeriesGenerator.add_noise(signal_econ_clean, snr_db=15)\n",
    "    print(f\"✓ Signal length: {len(signal_econ_clean)} samples\")\n",
    "    print(f\"  Range: [{signal_econ_clean.min():.2f}, {signal_econ_clean.max():.2f}]\")\n",
    "    \n",
    "    # ========== 2. ΠΡΟΕΤΟΙΜΑΣΙΑ DATASETS ==========\n",
    "    print(\"\\n\" + \"=\"*80)\n",
    "    print(\"ΦΑΣΗ 2: ΠΡΟΕΤΟΙΜΑΣΙΑ DATASETS\")\n",
    "    print(\"=\"*80)\n",
    "    \n",
    "    window_size = 64\n",
    "    batch_size = 32\n",
    "    \n",
    "    # Gravitational wave dataset\n",
    "    print(f\"\\n[2.1] Δημιουργία dataset για GW signal (window_size={window_size})...\")\n",
    "    X_gw_clean, y_gw_clean = TimeSeriesGenerator.create_dataset(\n",
    "        signal_gw_clean, window_size=window_size, stride=8\n",
    "    )\n",
    "    X_gw_noisy, _ = TimeSeriesGenerator.create_dataset(\n",
    "        signal_gw_noisy, window_size=window_size, stride=8\n",
    "    )\n",
    "    \n",
    "    # Split train/val/test\n",
    "    n_train = int(0.6 * len(X_gw_clean))\n",
    "    n_val = int(0.2 * len(X_gw_clean))\n",
    "    \n",
    "    X_gw_train_clean = X_gw_clean[:n_train]\n",
    "    X_gw_train_noisy = X_gw_noisy[:n_train]\n",
    "    X_gw_val_clean = X_gw_clean[n_train:n_train+n_val]\n",
    "    X_gw_val_noisy = X_gw_noisy[n_train:n_train+n_val]\n",
    "    X_gw_test_clean = X_gw_clean[n_train+n_val:]\n",
    "    X_gw_test_noisy = X_gw_noisy[n_train+n_val:]\n",
    "    \n",
    "    print(f\"✓ Train: {len(X_gw_train_clean)}, Val: {len(X_gw_val_clean)}, Test: {len(X_gw_test_clean)}\")\n",
    "    \n",
    "    # Create PyTorch DataLoaders\n",
    "    train_ds_gw = DenoisingDataset(X_gw_train_noisy, X_gw_train_clean)\n",
    "    val_ds_gw = DenoisingDataset(X_gw_val_noisy, X_gw_val_clean)\n",
    "    test_ds_gw = DenoisingDataset(X_gw_test_noisy, X_gw_test_clean)\n",
    "    \n",
    "    train_loader_gw = DataLoader(train_ds_gw, batch_size=batch_size, shuffle=True)\n",
    "    val_loader_gw = DataLoader(val_ds_gw, batch_size=batch_size, shuffle=False)\n",
    "    test_loader_gw = DataLoader(test_ds_gw, batch_size=batch_size, shuffle=False)\n",
    "    \n",
    "    # Economic signal dataset\n",
    "    print(f\"\\n[2.2] Δημιουργία dataset για economic signal...\")\n",
    "    X_econ_clean, y_econ_clean = TimeSeriesGenerator.create_dataset(\n",
    "        signal_econ_clean, window_size=window_size, stride=8\n",
    "    )\n",
    "    X_econ_noisy, _ = TimeSeriesGenerator.create_dataset(\n",
    "        signal_econ_noisy, window_size=window_size, stride=8\n",
    "    )\n",
    "    \n",
    "    # Normalize\n",
    "    scaler = StandardScaler()\n",
    "    X_econ_clean_flat = X_econ_clean.reshape(-1, window_size)\n",
    "    X_econ_clean_flat = scaler.fit_transform(X_econ_clean_flat)\n",
    "    X_econ_clean = X_econ_clean_flat.reshape(-1, window_size)\n",
    "    \n",
    "    X_econ_noisy_flat = X_econ_noisy.reshape(-1, window_size)\n",
    "    X_econ_noisy_flat = scaler.transform(X_econ_noisy_flat)\n",
    "    X_econ_noisy = X_econ_noisy_flat.reshape(-1, window_size)\n",
    "    \n",
    "    n_train_econ = int(0.6 * len(X_econ_clean))\n",
    "    n_val_econ = int(0.2 * len(X_econ_clean))\n",
    "    \n",
    "    X_econ_train_clean = X_econ_clean[:n_train_econ]\n",
    "    X_econ_train_noisy = X_econ_noisy[:n_train_econ]\n",
    "    X_econ_val_clean = X_econ_clean[n_train_econ:n_train_econ+n_val_econ]\n",
    "    X_econ_val_noisy = X_econ_noisy[n_train_econ:n_train_econ+n_val_econ]\n",
    "    X_econ_test_clean = X_econ_clean[n_train_econ+n_val_econ:]\n",
    "    X_econ_test_noisy = X_econ_noisy[n_train_econ+n_val_econ:]\n",
    "    \n",
    "    print(f\"✓ Train: {len(X_econ_train_clean)}, Val: {len(X_econ_val_clean)}, Test: {len(X_econ_test_clean)}\")\n",
    "    \n",
    "    # Create PyTorch DataLoaders\n",
    "    train_ds_econ = DenoisingDataset(X_econ_train_noisy, X_econ_train_clean)\n",
    "    val_ds_econ = DenoisingDataset(X_econ_val_noisy, X_econ_val_clean)\n",
    "    test_ds_econ = DenoisingDataset(X_econ_test_noisy, X_econ_test_clean)\n",
    "    \n",
    "    train_loader_econ = DataLoader(train_ds_econ, batch_size=batch_size, shuffle=True)\n",
    "    val_loader_econ = DataLoader(val_ds_econ, batch_size=batch_size, shuffle=False)\n",
    "    test_loader_econ = DataLoader(test_ds_econ, batch_size=batch_size, shuffle=False)\n",
    "    \n",
    "    # ========== 3. ΕΚΠΑΙΔΕΥΣΗ ΜΟΝΤΕΛΩΝ ==========\n",
    "    print(\"\\n\" + \"=\"*80)\n",
    "    print(\"ΦΑΣΗ 3: ΕΚΠΑΙΔΕΥΣΗ ΝΕΥΡΩΝΙΚΩΝ ΔΙΚΤΥΩΝ (PyTorch)\")\n",
    "    print(\"=\"*80)\n",
    "    \n",
    "    # Gravitational Wave: CNN Autoencoder\n",
    "    print(\"\\n[3.1] GRAVITATIONAL WAVE - CNN Autoencoder\")\n",
    "    model_gw_cnn = Conv1DAutoencoder(input_length=window_size, latent_dim=16)\n",
    "    train_denoising_model(\n",
    "        model_gw_cnn, train_loader_gw, val_loader_gw,\n",
    "        epochs=50, model_name=\"CNN-Autoencoder (GW)\", device=device\n",
    "    )\n",
    "    \n",
    "    # Gravitational Wave: TDNN\n",
    "    print(\"\\n[3.2] GRAVITATIONAL WAVE - TDNN\")\n",
    "    model_gw_tdnn = TDNN(input_length=window_size, time_delays=(1, 2, 4, 8))\n",
    "    train_denoising_model(\n",
    "        model_gw_tdnn, train_loader_gw, val_loader_gw,\n",
    "        epochs=50, model_name=\"TDNN (GW)\", device=device\n",
    "    )\n",
    "    \n",
    "    # Economic: CNN Autoencoder\n",
    "    print(\"\\n[3.3] ECONOMIC SIGNAL - CNN Autoencoder\")\n",
    "    model_econ_cnn = Conv1DAutoencoder(input_length=window_size, latent_dim=16)\n",
    "    train_denoising_model(\n",
    "        model_econ_cnn, train_loader_econ, val_loader_econ,\n",
    "        epochs=100, model_name=\"CNN-Autoencoder (Economic)\", device=device\n",
    "    )\n",
    "    \n",
    "    # Economic: TDNN\n",
    "    print(\"\\n[3.4] ECONOMIC SIGNAL - TDNN\")\n",
    "    model_econ_tdnn = TDNN(input_length=window_size, time_delays=(1, 2, 4, 8))\n",
    "    train_denoising_model(\n",
    "        model_econ_tdnn, train_loader_econ, val_loader_econ,\n",
    "        epochs=100, model_name=\"TDNN (Economic)\", device=device\n",
    "    )\n",
    "    \n",
    "    # ========== 4. ΑΞΙΟΛΟΓΗΣΗ ΜΟΝΤΕΛΩΝ ==========\n",
    "    print(\"\\n\" + \"=\"*80)\n",
    "    print(\"ΦΑΣΗ 4: ΑΞΙΟΛΟΓΗΣΗ ΜΟΝΤΕΛΩΝ\")\n",
    "    print(\"=\"*80)\n",
    "    \n",
    "    results_gw_cnn = evaluate_denoiser(model_gw_cnn, test_loader_gw, \"CNN-Autoencoder (GW)\", device=device)\n",
    "    results_gw_tdnn = evaluate_denoiser(model_gw_tdnn, test_loader_gw, \"TDNN (GW)\", device=device)\n",
    "    results_econ_cnn = evaluate_denoiser(model_econ_cnn, test_loader_econ, \"CNN-Autoencoder (Economic)\", device=device)\n",
    "    results_econ_tdnn = evaluate_denoiser(model_econ_tdnn, test_loader_econ, \"TDNN (Economic)\", device=device)\n",
    "    \n",
    "    # ========== 5. FULL SIGNAL PREDICTIONS ==========\n",
    "    print(\"\\n\" + \"=\"*80)\n",
    "    print(\"ΦΑΣΗ 5: PREDICTIONS ΣΕ ΟΛΟΚΛΗΡΟ ΤΟ ΣΗΜΑ\")\n",
    "    print(\"=\"*80)\n",
    "    \n",
    "    print(\"\\n[5.1] Prediction για Gravitational Wave...\")\n",
    "    gw_pred_cnn = predict_full_signal(model_gw_cnn, signal_gw_noisy, window_size, stride=8, device=device)\n",
    "    gw_pred_tdnn = predict_full_signal(model_gw_tdnn, signal_gw_noisy, window_size, stride=8, device=device)\n",
    "    print(\"✓ Completed\")\n",
    "    \n",
    "    print(\"\\n[5.2] Prediction για Economic Signal...\")\n",
    "    econ_pred_cnn = predict_full_signal(model_econ_cnn, signal_econ_noisy, window_size, stride=8, device=device)\n",
    "    econ_pred_tdnn = predict_full_signal(model_econ_tdnn, signal_econ_noisy, window_size, stride=8, device=device)\n",
    "    print(\"✓ Completed\")\n",
    "    \n",
    "    # ========== 6. EVENT DETECTION ==========\n",
    "    print(\"\\n\" + \"=\"*80)\n",
    "    print(\"ΦΑΣΗ 6: ΑΝΙΧΝΕΥΣΗ ΣΥΜΒΑΝΤΩΝ (EVENT DETECTION)\")\n",
    "    print(\"=\"*80)\n",
    "    \n",
    "    print(\"\\n[6.1] GRAVITATIONAL WAVE - Event Detection\")\n",
    "    events_gw = detect_events(signal_gw_clean, signal_gw_noisy, gw_pred_cnn, threshold_std=1.5)\n",
    "    \n",
    "    print(f\"\\nClean signal events: {len(events_gw['clean'])}\")\n",
    "    for i, event in enumerate(events_gw['clean'][:3]):\n",
    "        print(f\"  Event {i+1}: center={event['center']:.1f}, \"\n",
    "              f\"amplitude={event['amplitude']:.4f}, max_acc={event['max_acceleration']:.4f}\")\n",
    "    \n",
    "    print(f\"\\nNoisy signal events: {len(events_gw['noisy'])}\")\n",
    "    print(f\"Denoised (CNN) signal events: {len(events_gw['denoised'])}\")\n",
    "    \n",
    "    if len(events_gw['clean']) > 0:\n",
    "        detected = len(events_gw['denoised'])\n",
    "        true_events = len(events_gw['clean'])\n",
    "        recall = min(detected / true_events, 1.0)\n",
    "        print(f\"\\nEvent Detection Recall: {recall*100:.1f}%\")\n",
    "    \n",
    "    # Economic events\n",
    "    print(\"\\n[6.2] ECONOMIC SIGNAL - Crisis Detection\")\n",
    "    events_econ = detect_events(signal_econ_clean, signal_econ_noisy, econ_pred_cnn, threshold_std=1.0)\n",
    "    \n",
    "    print(f\"Clean signal crises: {len(events_econ['clean'])}\")\n",
    "    print(f\"Noisy signal crises: {len(events_econ['noisy'])}\")\n",
    "    print(f\"Denoised signal crises: {len(events_econ['denoised'])}\")\n",
    "    \n",
    "    # ========== 7. ΑΝΑΛΥΣΗ - ΣΥΜΠΕΡΑΣΜΑΤΑ ==========\n",
    "    print(\"\\n\" + \"=\"*80)\n",
    "    print(\"ΦΑΣΗ 7: ΑΝΑΛΥΣΗ - ΣΥΜΠΕΡΑΣΜΑΤΑ\")\n",
    "    print(\"=\"*80)\n",
    "    \n",
    "    print(\"\\n[GRAVITATIONAL WAVES]\")\n",
    "    print(f\"✓ CNN Autoencoder: SNR improvement = {results_gw_cnn['snr_improvement']:.2f} dB\")\n",
    "    print(f\"✓ TDNN: SNR improvement = {results_gw_tdnn['snr_improvement']:.2f} dB\")\n",
    "    best_gw = 'CNN' if results_gw_cnn['snr_improvement'] > results_gw_tdnn['snr_improvement'] else 'TDNN'\n",
    "    print(f\"➜ Καλύτερο μοντέλο: {best_gw}\")\n",
    "    if len(events_gw['clean']) > 0:\n",
    "        print(f\"  Event detection success: {len(events_gw['denoised']) / len(events_gw['clean']) * 100:.1f}%\")\n",
    "    \n",
    "    print(\"\\n[ECONOMIC SIGNALS]\")\n",
    "    print(f\"✓ CNN Autoencoder: SNR improvement = {results_econ_cnn['snr_improvement']:.2f} dB\")\n",
    "    print(f\"✓ TDNN: SNR improvement = {results_econ_tdnn['snr_improvement']:.2f} dB\")\n",
    "    best_econ = 'CNN' if results_econ_cnn['snr_improvement'] > results_econ_tdnn['snr_improvement'] else 'TDNN'\n",
    "    print(f\"➜ Καλύτερο μοντέλο: {best_econ}\")\n",
    "    if len(events_econ['clean']) > 0:\n",
    "        print(f\"  Crisis detection success: {len(events_econ['denoised']) / len(events_econ['clean']) * 100:.1f}%\")\n",
    "    \n",
    "    # ========== 8. ΕΞΟΔΟΣ ΔΕΔΟΜΕΝΩΝ ==========\n",
    "    print(\"\\n\" + \"=\"*80)\n",
    "    print(\"ΦΑΣΗ 8: ΕΞΟΔΟΣ ΑΠΟΤΕΛΕΣΜΑΤΩΝ\")\n",
    "    print(\"=\"*80)\n",
    "    \n",
    "    # Αποθήκευση\n",
    "    np.savez('denoising_results_pytorch.npz', \n",
    "             signal_gw_clean=signal_gw_clean,\n",
    "             signal_gw_noisy=signal_gw_noisy,\n",
    "             signal_gw_denoised=gw_pred_cnn,\n",
    "             signal_econ_clean=signal_econ_clean,\n",
    "             signal_econ_noisy=signal_econ_noisy,\n",
    "             signal_econ_denoised=econ_pred_cnn)\n",
    "    \n",
    "    print(\"✓ Αποτελέσματα αποθηκεύτηκαν στο denoising_results_pytorch.npz\")\n",
    "    \n",
    "    return {\n",
    "        'gw_clean': signal_gw_clean,\n",
    "        'gw_noisy': signal_gw_noisy,\n",
    "        'gw_denoised': gw_pred_cnn,\n",
    "        'econ_clean': signal_econ_clean,\n",
    "        'econ_noisy': signal_econ_noisy,\n",
    "        'econ_denoised': econ_pred_cnn,\n",
    "        'models': {\n",
    "            'gw_cnn': model_gw_cnn,\n",
    "            'gw_tdnn': model_gw_tdnn,\n",
    "            'econ_cnn': model_econ_cnn,\n",
    "            'econ_tdnn': model_econ_tdnn\n",
    "        }\n",
    "    }\n",
    " \n",
    " \n",
    "if __name__ == \"__main__\":\n",
    "    data = main()\n",
    "    print(\"\\n\" + \"=\"*80)\n",
    "    print(\"ΟΛΟΚΛΗΡΩΣΗ ΑΝΑΛΥΣΗΣ (PYTORCH)\")\n",
    "    print(\"=\"*80)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 44,
   "id": "2337f0c6",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Using device: cuda\n",
      "================================================================================\n",
      "TIME SERIES DENOISING WITH PYTORCH - REAL FINANCIAL DATA\n",
      "================================================================================\n",
      "\n",
      "================================================================================\n",
      "ΦΑΣΗ 1: ΚΑΤΕΒΑΣΜΑ ΠΡΑΓΜΑΤΙΚΩΝ ΔΕΔΟΜΕΝΩΝ\n",
      "================================================================================\n",
      "\n",
      "[1.1] Κατέβασμα δεδομένων S&P 500...\n",
      "\n",
      "[📊] Κατέβασμα δεδομένων για ^GSPC (2019-01-01 → 2024-12-31)...\n",
      "✓ 1509 κλεισίματα κατεβασμένα για ^GSPC\n",
      "\n",
      "[1.2] Ανίχνευση περιόδων κρίσης στα δεδομένα...\n",
      "✓ 1509 ημέρες καταγραφής\n",
      "✓ Μέση ημερήσια volatility: nan\n",
      "✓ Περιόδους κρίσης ανιχνευμένες: 0\n",
      "\n",
      "[1.3] Κατέβασμα δεδομένων Apple (AAPL)...\n",
      "\n",
      "[📊] Κατέβασμα δεδομένων για AAPL (2019-01-01 → 2024-12-31)...\n",
      "✓ 1509 κλεισίματα κατεβασμένα για AAPL\n",
      "✓ Μέσο return: nan\n",
      "✓ Volatility: nan\n",
      "\n",
      "================================================================================\n",
      "ΦΑΣΗ 2: ΠΡΟΕΤΟΙΜΑΣΙΑ ΔΕΔΟΜΕΝΩΝ\n",
      "================================================================================\n"
     ]
    },
    {
     "ename": "ValueError",
     "evalue": "Found array with 0 sample(s) (shape=(0, 1)) while a minimum of 1 is required by StandardScaler.",
     "output_type": "error",
     "traceback": [
      "\u001b[1;31m---------------------------------------------------------------------------\u001b[0m",
      "\u001b[1;31mValueError\u001b[0m                                Traceback (most recent call last)",
      "Cell \u001b[1;32mIn[44], line 668\u001b[0m\n\u001b[0;32m    654\u001b[0m     \u001b[38;5;28;01mreturn\u001b[39;00m {\n\u001b[0;32m    655\u001b[0m         \u001b[38;5;124m'\u001b[39m\u001b[38;5;124msignal_clean\u001b[39m\u001b[38;5;124m'\u001b[39m: signal_clean_normalized,\n\u001b[0;32m    656\u001b[0m         \u001b[38;5;124m'\u001b[39m\u001b[38;5;124msignal_noisy\u001b[39m\u001b[38;5;124m'\u001b[39m: signal_noisy,\n\u001b[1;32m   (...)\u001b[0m\n\u001b[0;32m    663\u001b[0m         \u001b[38;5;124m'\u001b[39m\u001b[38;5;124mdf_apple\u001b[39m\u001b[38;5;124m'\u001b[39m: df_apple \u001b[38;5;28;01mif\u001b[39;00m df_apple \u001b[38;5;129;01mis\u001b[39;00m \u001b[38;5;129;01mnot\u001b[39;00m \u001b[38;5;28;01mNone\u001b[39;00m \u001b[38;5;28;01melse\u001b[39;00m \u001b[38;5;28;01mNone\u001b[39;00m\n\u001b[0;32m    664\u001b[0m     }\n\u001b[0;32m    667\u001b[0m \u001b[38;5;28;01mif\u001b[39;00m \u001b[38;5;18m__name__\u001b[39m \u001b[38;5;241m==\u001b[39m \u001b[38;5;124m\"\u001b[39m\u001b[38;5;124m__main__\u001b[39m\u001b[38;5;124m\"\u001b[39m:\n\u001b[1;32m--> 668\u001b[0m     results \u001b[38;5;241m=\u001b[39m \u001b[43mmain\u001b[49m\u001b[43m(\u001b[49m\u001b[43m)\u001b[49m\n\u001b[0;32m    669\u001b[0m     \u001b[38;5;28mprint\u001b[39m(\u001b[38;5;124m\"\u001b[39m\u001b[38;5;130;01m\\n\u001b[39;00m\u001b[38;5;124m\"\u001b[39m \u001b[38;5;241m+\u001b[39m \u001b[38;5;124m\"\u001b[39m\u001b[38;5;124m=\u001b[39m\u001b[38;5;124m\"\u001b[39m\u001b[38;5;241m*\u001b[39m\u001b[38;5;241m80\u001b[39m)\n\u001b[0;32m    670\u001b[0m     \u001b[38;5;28mprint\u001b[39m(\u001b[38;5;124m\"\u001b[39m\u001b[38;5;124mΟΛΟΚΛΗΡΩΣΗ ΑΝΑΛΥΣΗΣ\u001b[39m\u001b[38;5;124m\"\u001b[39m)\n",
      "Cell \u001b[1;32mIn[44], line 496\u001b[0m, in \u001b[0;36mmain\u001b[1;34m()\u001b[0m\n\u001b[0;32m    494\u001b[0m \u001b[38;5;66;03m# Normalize\u001b[39;00m\n\u001b[0;32m    495\u001b[0m scaler \u001b[38;5;241m=\u001b[39m StandardScaler()\n\u001b[1;32m--> 496\u001b[0m signal_clean_normalized \u001b[38;5;241m=\u001b[39m \u001b[43mscaler\u001b[49m\u001b[38;5;241;43m.\u001b[39;49m\u001b[43mfit_transform\u001b[49m\u001b[43m(\u001b[49m\u001b[43msignal_clean\u001b[49m\u001b[38;5;241;43m.\u001b[39;49m\u001b[43mreshape\u001b[49m\u001b[43m(\u001b[49m\u001b[38;5;241;43m-\u001b[39;49m\u001b[38;5;241;43m1\u001b[39;49m\u001b[43m,\u001b[49m\u001b[43m \u001b[49m\u001b[38;5;241;43m1\u001b[39;49m\u001b[43m)\u001b[49m\u001b[43m)\u001b[49m\u001b[38;5;241m.\u001b[39mflatten()\n\u001b[0;32m    498\u001b[0m \u001b[38;5;66;03m# Προσθήκη noise\u001b[39;00m\n\u001b[0;32m    499\u001b[0m \u001b[38;5;28mprint\u001b[39m(\u001b[38;5;124m\"\u001b[39m\u001b[38;5;130;01m\\n\u001b[39;00m\u001b[38;5;124m[2.1] Προσθήκη Gaussian noise (SNR=15dB)...\u001b[39m\u001b[38;5;124m\"\u001b[39m)\n",
      "File \u001b[1;32mc:\\Users\\vasil\\anaconda3\\envs\\pytorch_env\\Lib\\site-packages\\sklearn\\utils\\_set_output.py:313\u001b[0m, in \u001b[0;36m_wrap_method_output.<locals>.wrapped\u001b[1;34m(self, X, *args, **kwargs)\u001b[0m\n\u001b[0;32m    311\u001b[0m \u001b[38;5;129m@wraps\u001b[39m(f)\n\u001b[0;32m    312\u001b[0m \u001b[38;5;28;01mdef\u001b[39;00m \u001b[38;5;21mwrapped\u001b[39m(\u001b[38;5;28mself\u001b[39m, X, \u001b[38;5;241m*\u001b[39margs, \u001b[38;5;241m*\u001b[39m\u001b[38;5;241m*\u001b[39mkwargs):\n\u001b[1;32m--> 313\u001b[0m     data_to_wrap \u001b[38;5;241m=\u001b[39m \u001b[43mf\u001b[49m\u001b[43m(\u001b[49m\u001b[38;5;28;43mself\u001b[39;49m\u001b[43m,\u001b[49m\u001b[43m \u001b[49m\u001b[43mX\u001b[49m\u001b[43m,\u001b[49m\u001b[43m \u001b[49m\u001b[38;5;241;43m*\u001b[39;49m\u001b[43margs\u001b[49m\u001b[43m,\u001b[49m\u001b[43m \u001b[49m\u001b[38;5;241;43m*\u001b[39;49m\u001b[38;5;241;43m*\u001b[39;49m\u001b[43mkwargs\u001b[49m\u001b[43m)\u001b[49m\n\u001b[0;32m    314\u001b[0m     \u001b[38;5;28;01mif\u001b[39;00m \u001b[38;5;28misinstance\u001b[39m(data_to_wrap, \u001b[38;5;28mtuple\u001b[39m):\n\u001b[0;32m    315\u001b[0m         \u001b[38;5;66;03m# only wrap the first output for cross decomposition\u001b[39;00m\n\u001b[0;32m    316\u001b[0m         return_tuple \u001b[38;5;241m=\u001b[39m (\n\u001b[0;32m    317\u001b[0m             _wrap_data_with_container(method, data_to_wrap[\u001b[38;5;241m0\u001b[39m], X, \u001b[38;5;28mself\u001b[39m),\n\u001b[0;32m    318\u001b[0m             \u001b[38;5;241m*\u001b[39mdata_to_wrap[\u001b[38;5;241m1\u001b[39m:],\n\u001b[0;32m    319\u001b[0m         )\n",
      "File \u001b[1;32mc:\\Users\\vasil\\anaconda3\\envs\\pytorch_env\\Lib\\site-packages\\sklearn\\base.py:1098\u001b[0m, in \u001b[0;36mTransformerMixin.fit_transform\u001b[1;34m(self, X, y, **fit_params)\u001b[0m\n\u001b[0;32m   1083\u001b[0m         warnings\u001b[38;5;241m.\u001b[39mwarn(\n\u001b[0;32m   1084\u001b[0m             (\n\u001b[0;32m   1085\u001b[0m                 \u001b[38;5;124mf\u001b[39m\u001b[38;5;124m\"\u001b[39m\u001b[38;5;124mThis object (\u001b[39m\u001b[38;5;132;01m{\u001b[39;00m\u001b[38;5;28mself\u001b[39m\u001b[38;5;241m.\u001b[39m\u001b[38;5;18m__class__\u001b[39m\u001b[38;5;241m.\u001b[39m\u001b[38;5;18m__name__\u001b[39m\u001b[38;5;132;01m}\u001b[39;00m\u001b[38;5;124m) has a `transform`\u001b[39m\u001b[38;5;124m\"\u001b[39m\n\u001b[1;32m   (...)\u001b[0m\n\u001b[0;32m   1093\u001b[0m             \u001b[38;5;167;01mUserWarning\u001b[39;00m,\n\u001b[0;32m   1094\u001b[0m         )\n\u001b[0;32m   1096\u001b[0m \u001b[38;5;28;01mif\u001b[39;00m y \u001b[38;5;129;01mis\u001b[39;00m \u001b[38;5;28;01mNone\u001b[39;00m:\n\u001b[0;32m   1097\u001b[0m     \u001b[38;5;66;03m# fit method of arity 1 (unsupervised transformation)\u001b[39;00m\n\u001b[1;32m-> 1098\u001b[0m     \u001b[38;5;28;01mreturn\u001b[39;00m \u001b[38;5;28;43mself\u001b[39;49m\u001b[38;5;241;43m.\u001b[39;49m\u001b[43mfit\u001b[49m\u001b[43m(\u001b[49m\u001b[43mX\u001b[49m\u001b[43m,\u001b[49m\u001b[43m \u001b[49m\u001b[38;5;241;43m*\u001b[39;49m\u001b[38;5;241;43m*\u001b[39;49m\u001b[43mfit_params\u001b[49m\u001b[43m)\u001b[49m\u001b[38;5;241m.\u001b[39mtransform(X)\n\u001b[0;32m   1099\u001b[0m \u001b[38;5;28;01melse\u001b[39;00m:\n\u001b[0;32m   1100\u001b[0m     \u001b[38;5;66;03m# fit method of arity 2 (supervised transformation)\u001b[39;00m\n\u001b[0;32m   1101\u001b[0m     \u001b[38;5;28;01mreturn\u001b[39;00m \u001b[38;5;28mself\u001b[39m\u001b[38;5;241m.\u001b[39mfit(X, y, \u001b[38;5;241m*\u001b[39m\u001b[38;5;241m*\u001b[39mfit_params)\u001b[38;5;241m.\u001b[39mtransform(X)\n",
      "File \u001b[1;32mc:\\Users\\vasil\\anaconda3\\envs\\pytorch_env\\Lib\\site-packages\\sklearn\\preprocessing\\_data.py:878\u001b[0m, in \u001b[0;36mStandardScaler.fit\u001b[1;34m(self, X, y, sample_weight)\u001b[0m\n\u001b[0;32m    876\u001b[0m \u001b[38;5;66;03m# Reset internal state before fitting\u001b[39;00m\n\u001b[0;32m    877\u001b[0m \u001b[38;5;28mself\u001b[39m\u001b[38;5;241m.\u001b[39m_reset()\n\u001b[1;32m--> 878\u001b[0m \u001b[38;5;28;01mreturn\u001b[39;00m \u001b[38;5;28;43mself\u001b[39;49m\u001b[38;5;241;43m.\u001b[39;49m\u001b[43mpartial_fit\u001b[49m\u001b[43m(\u001b[49m\u001b[43mX\u001b[49m\u001b[43m,\u001b[49m\u001b[43m \u001b[49m\u001b[43my\u001b[49m\u001b[43m,\u001b[49m\u001b[43m \u001b[49m\u001b[43msample_weight\u001b[49m\u001b[43m)\u001b[49m\n",
      "File \u001b[1;32mc:\\Users\\vasil\\anaconda3\\envs\\pytorch_env\\Lib\\site-packages\\sklearn\\base.py:1473\u001b[0m, in \u001b[0;36m_fit_context.<locals>.decorator.<locals>.wrapper\u001b[1;34m(estimator, *args, **kwargs)\u001b[0m\n\u001b[0;32m   1466\u001b[0m     estimator\u001b[38;5;241m.\u001b[39m_validate_params()\n\u001b[0;32m   1468\u001b[0m \u001b[38;5;28;01mwith\u001b[39;00m config_context(\n\u001b[0;32m   1469\u001b[0m     skip_parameter_validation\u001b[38;5;241m=\u001b[39m(\n\u001b[0;32m   1470\u001b[0m         prefer_skip_nested_validation \u001b[38;5;129;01mor\u001b[39;00m global_skip_validation\n\u001b[0;32m   1471\u001b[0m     )\n\u001b[0;32m   1472\u001b[0m ):\n\u001b[1;32m-> 1473\u001b[0m     \u001b[38;5;28;01mreturn\u001b[39;00m \u001b[43mfit_method\u001b[49m\u001b[43m(\u001b[49m\u001b[43mestimator\u001b[49m\u001b[43m,\u001b[49m\u001b[43m \u001b[49m\u001b[38;5;241;43m*\u001b[39;49m\u001b[43margs\u001b[49m\u001b[43m,\u001b[49m\u001b[43m \u001b[49m\u001b[38;5;241;43m*\u001b[39;49m\u001b[38;5;241;43m*\u001b[39;49m\u001b[43mkwargs\u001b[49m\u001b[43m)\u001b[49m\n",
      "File \u001b[1;32mc:\\Users\\vasil\\anaconda3\\envs\\pytorch_env\\Lib\\site-packages\\sklearn\\preprocessing\\_data.py:914\u001b[0m, in \u001b[0;36mStandardScaler.partial_fit\u001b[1;34m(self, X, y, sample_weight)\u001b[0m\n\u001b[0;32m    882\u001b[0m \u001b[38;5;250m\u001b[39m\u001b[38;5;124;03m\"\"\"Online computation of mean and std on X for later scaling.\u001b[39;00m\n\u001b[0;32m    883\u001b[0m \n\u001b[0;32m    884\u001b[0m \u001b[38;5;124;03mAll of X is processed as a single batch. This is intended for cases\u001b[39;00m\n\u001b[1;32m   (...)\u001b[0m\n\u001b[0;32m    911\u001b[0m \u001b[38;5;124;03m    Fitted scaler.\u001b[39;00m\n\u001b[0;32m    912\u001b[0m \u001b[38;5;124;03m\"\"\"\u001b[39;00m\n\u001b[0;32m    913\u001b[0m first_call \u001b[38;5;241m=\u001b[39m \u001b[38;5;129;01mnot\u001b[39;00m \u001b[38;5;28mhasattr\u001b[39m(\u001b[38;5;28mself\u001b[39m, \u001b[38;5;124m\"\u001b[39m\u001b[38;5;124mn_samples_seen_\u001b[39m\u001b[38;5;124m\"\u001b[39m)\n\u001b[1;32m--> 914\u001b[0m X \u001b[38;5;241m=\u001b[39m \u001b[38;5;28;43mself\u001b[39;49m\u001b[38;5;241;43m.\u001b[39;49m\u001b[43m_validate_data\u001b[49m\u001b[43m(\u001b[49m\n\u001b[0;32m    915\u001b[0m \u001b[43m    \u001b[49m\u001b[43mX\u001b[49m\u001b[43m,\u001b[49m\n\u001b[0;32m    916\u001b[0m \u001b[43m    \u001b[49m\u001b[43maccept_sparse\u001b[49m\u001b[38;5;241;43m=\u001b[39;49m\u001b[43m(\u001b[49m\u001b[38;5;124;43m\"\u001b[39;49m\u001b[38;5;124;43mcsr\u001b[39;49m\u001b[38;5;124;43m\"\u001b[39;49m\u001b[43m,\u001b[49m\u001b[43m \u001b[49m\u001b[38;5;124;43m\"\u001b[39;49m\u001b[38;5;124;43mcsc\u001b[39;49m\u001b[38;5;124;43m\"\u001b[39;49m\u001b[43m)\u001b[49m\u001b[43m,\u001b[49m\n\u001b[0;32m    917\u001b[0m \u001b[43m    \u001b[49m\u001b[43mdtype\u001b[49m\u001b[38;5;241;43m=\u001b[39;49m\u001b[43mFLOAT_DTYPES\u001b[49m\u001b[43m,\u001b[49m\n\u001b[0;32m    918\u001b[0m \u001b[43m    \u001b[49m\u001b[43mforce_all_finite\u001b[49m\u001b[38;5;241;43m=\u001b[39;49m\u001b[38;5;124;43m\"\u001b[39;49m\u001b[38;5;124;43mallow-nan\u001b[39;49m\u001b[38;5;124;43m\"\u001b[39;49m\u001b[43m,\u001b[49m\n\u001b[0;32m    919\u001b[0m \u001b[43m    \u001b[49m\u001b[43mreset\u001b[49m\u001b[38;5;241;43m=\u001b[39;49m\u001b[43mfirst_call\u001b[49m\u001b[43m,\u001b[49m\n\u001b[0;32m    920\u001b[0m \u001b[43m\u001b[49m\u001b[43m)\u001b[49m\n\u001b[0;32m    921\u001b[0m n_features \u001b[38;5;241m=\u001b[39m X\u001b[38;5;241m.\u001b[39mshape[\u001b[38;5;241m1\u001b[39m]\n\u001b[0;32m    923\u001b[0m \u001b[38;5;28;01mif\u001b[39;00m sample_weight \u001b[38;5;129;01mis\u001b[39;00m \u001b[38;5;129;01mnot\u001b[39;00m \u001b[38;5;28;01mNone\u001b[39;00m:\n",
      "File \u001b[1;32mc:\\Users\\vasil\\anaconda3\\envs\\pytorch_env\\Lib\\site-packages\\sklearn\\base.py:633\u001b[0m, in \u001b[0;36mBaseEstimator._validate_data\u001b[1;34m(self, X, y, reset, validate_separately, cast_to_ndarray, **check_params)\u001b[0m\n\u001b[0;32m    631\u001b[0m         out \u001b[38;5;241m=\u001b[39m X, y\n\u001b[0;32m    632\u001b[0m \u001b[38;5;28;01melif\u001b[39;00m \u001b[38;5;129;01mnot\u001b[39;00m no_val_X \u001b[38;5;129;01mand\u001b[39;00m no_val_y:\n\u001b[1;32m--> 633\u001b[0m     out \u001b[38;5;241m=\u001b[39m \u001b[43mcheck_array\u001b[49m\u001b[43m(\u001b[49m\u001b[43mX\u001b[49m\u001b[43m,\u001b[49m\u001b[43m \u001b[49m\u001b[43minput_name\u001b[49m\u001b[38;5;241;43m=\u001b[39;49m\u001b[38;5;124;43m\"\u001b[39;49m\u001b[38;5;124;43mX\u001b[39;49m\u001b[38;5;124;43m\"\u001b[39;49m\u001b[43m,\u001b[49m\u001b[43m \u001b[49m\u001b[38;5;241;43m*\u001b[39;49m\u001b[38;5;241;43m*\u001b[39;49m\u001b[43mcheck_params\u001b[49m\u001b[43m)\u001b[49m\n\u001b[0;32m    634\u001b[0m \u001b[38;5;28;01melif\u001b[39;00m no_val_X \u001b[38;5;129;01mand\u001b[39;00m \u001b[38;5;129;01mnot\u001b[39;00m no_val_y:\n\u001b[0;32m    635\u001b[0m     out \u001b[38;5;241m=\u001b[39m _check_y(y, \u001b[38;5;241m*\u001b[39m\u001b[38;5;241m*\u001b[39mcheck_params)\n",
      "File \u001b[1;32mc:\\Users\\vasil\\anaconda3\\envs\\pytorch_env\\Lib\\site-packages\\sklearn\\utils\\validation.py:1087\u001b[0m, in \u001b[0;36mcheck_array\u001b[1;34m(array, accept_sparse, accept_large_sparse, dtype, order, copy, force_writeable, force_all_finite, ensure_2d, allow_nd, ensure_min_samples, ensure_min_features, estimator, input_name)\u001b[0m\n\u001b[0;32m   1085\u001b[0m     n_samples \u001b[38;5;241m=\u001b[39m _num_samples(array)\n\u001b[0;32m   1086\u001b[0m     \u001b[38;5;28;01mif\u001b[39;00m n_samples \u001b[38;5;241m<\u001b[39m ensure_min_samples:\n\u001b[1;32m-> 1087\u001b[0m         \u001b[38;5;28;01mraise\u001b[39;00m \u001b[38;5;167;01mValueError\u001b[39;00m(\n\u001b[0;32m   1088\u001b[0m             \u001b[38;5;124m\"\u001b[39m\u001b[38;5;124mFound array with \u001b[39m\u001b[38;5;132;01m%d\u001b[39;00m\u001b[38;5;124m sample(s) (shape=\u001b[39m\u001b[38;5;132;01m%s\u001b[39;00m\u001b[38;5;124m) while a\u001b[39m\u001b[38;5;124m\"\u001b[39m\n\u001b[0;32m   1089\u001b[0m             \u001b[38;5;124m\"\u001b[39m\u001b[38;5;124m minimum of \u001b[39m\u001b[38;5;132;01m%d\u001b[39;00m\u001b[38;5;124m is required\u001b[39m\u001b[38;5;132;01m%s\u001b[39;00m\u001b[38;5;124m.\u001b[39m\u001b[38;5;124m\"\u001b[39m\n\u001b[0;32m   1090\u001b[0m             \u001b[38;5;241m%\u001b[39m (n_samples, array\u001b[38;5;241m.\u001b[39mshape, ensure_min_samples, context)\n\u001b[0;32m   1091\u001b[0m         )\n\u001b[0;32m   1093\u001b[0m \u001b[38;5;28;01mif\u001b[39;00m ensure_min_features \u001b[38;5;241m>\u001b[39m \u001b[38;5;241m0\u001b[39m \u001b[38;5;129;01mand\u001b[39;00m array\u001b[38;5;241m.\u001b[39mndim \u001b[38;5;241m==\u001b[39m \u001b[38;5;241m2\u001b[39m:\n\u001b[0;32m   1094\u001b[0m     n_features \u001b[38;5;241m=\u001b[39m array\u001b[38;5;241m.\u001b[39mshape[\u001b[38;5;241m1\u001b[39m]\n",
      "\u001b[1;31mValueError\u001b[0m: Found array with 0 sample(s) (shape=(0, 1)) while a minimum of 1 is required by StandardScaler."
     ]
    }
   ],
   "source": [
    "\"\"\"\n",
    "Ανάλυση χρονοσειρών με Denoising μέσω PyTorch 1D CNNs/TDNNs\n",
    "ΠΡΑΓΜΑΤΙΚΑ ΟΙΚΟΝΟΜΙΚΑ ΔΕΔΟΜΕΝΑ ΑΠΟ YFINANCE\n",
    "\n",
    "Σε αυτή την έκδοση:\n",
    "✓ Κατέβασμα πραγματικών δεδομένων S&P 500 & άλλων μετοχών\n",
    "✓ Ανάλυση περιόδων κρίσης στα πραγματικά δεδομένα\n",
    "✓ Denoising με CNN & TDNN\n",
    "✓ Event detection σε πραγματικές κρίσεις\n",
    "\"\"\"\n",
    "\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from scipy import signal\n",
    "from sklearn.preprocessing import StandardScaler\n",
    "from sklearn.metrics import mean_squared_error, mean_absolute_error\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# PyTorch\n",
    "import torch\n",
    "import torch.nn as nn\n",
    "import torch.optim as optim\n",
    "from torch.utils.data import Dataset, DataLoader\n",
    "import torch.nn.functional as F\n",
    "\n",
    "# yfinance για κατέβασμα δεδομένων\n",
    "import yfinance as yf\n",
    "from datetime import datetime, timedelta\n",
    "\n",
    "# Device setup\n",
    "device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n",
    "print(f\"Using device: {device}\")\n",
    "\n",
    "# Random seeds\n",
    "np.random.seed(42)\n",
    "torch.manual_seed(42)\n",
    "\n",
    "print(\"=\" * 80)\n",
    "print(\"TIME SERIES DENOISING WITH PYTORCH - REAL FINANCIAL DATA\")\n",
    "print(\"=\" * 80)\n",
    "\n",
    "# ============================================================================\n",
    "# ΤΜΗΜΑ 1: ΚΑΤΕΒΑΣΜΑ ΠΡΑΓΜΑΤΙΚΩΝ ΔΕΔΟΜΕΝΩΝ\n",
    "# ============================================================================\n",
    "\n",
    "class FinancialDataDownloader:\n",
    "    \"\"\"Κατέβασμα και επεξεργασία χρηματοοικονομικών δεδομένων από yfinance\"\"\"\n",
    "    \n",
    "    @staticmethod\n",
    "    def download_stock_data(ticker, start_date=None, end_date=None):\n",
    "        \"\"\"\n",
    "        Κατεβάζει ιστορικά δεδομένα για μετοχή.\n",
    "        \n",
    "        Args:\n",
    "            ticker: Σύμβολο μετοχής (π.χ. \"^GSPC\" για S&P 500)\n",
    "            start_date: Ημ/νία έναρξης (π.χ. \"2015-01-01\")\n",
    "            end_date: Ημ/νία λήξης\n",
    "        \n",
    "        Returns:\n",
    "            df: DataFrame με OHLCV δεδομένα\n",
    "        \"\"\"\n",
    "        \n",
    "        if start_date is None:\n",
    "            start_date = (datetime.now() - timedelta(days=365*5)).strftime(\"%Y-%m-%d\")\n",
    "        if end_date is None:\n",
    "            end_date = datetime.now().strftime(\"%Y-%m-%d\")\n",
    "        \n",
    "        print(f\"\\n[📊] Κατέβασμα δεδομένων για {ticker} ({start_date} → {end_date})...\")\n",
    "        \n",
    "        try:\n",
    "            df = yf.download(ticker, start=start_date, end=end_date, progress=False)\n",
    "            print(f\"✓ {len(df)} κλεισίματα κατεβασμένα για {ticker}\")\n",
    "            return df\n",
    "        except Exception as e:\n",
    "            print(f\"✗ Σφάλμα κατεβάσματος: {e}\")\n",
    "            return None\n",
    "    \n",
    "    @staticmethod\n",
    "    def calculate_log_returns(df, column='Close'):\n",
    "        \"\"\"\n",
    "        Υπολογίζει log-returns από τιμές κλεισίματος.\n",
    "        \n",
    "        Log-returns = ln(P_t / P_{t-1})\n",
    "        \n",
    "        Πλεονεκτήματα:\n",
    "        ✓ Symmetrical (loss και gain αντιμετρικά)\n",
    "        ✓ Additive (συνδυάζονται εύκολα)\n",
    "        ✓ Κανονικότερη κατανομή από απλές returns\n",
    "        \"\"\"\n",
    "        \n",
    "        prices = df[column].values\n",
    "        log_returns = np.diff(np.log(prices))\n",
    "        \n",
    "        return log_returns\n",
    "    \n",
    "    @staticmethod\n",
    "    def identify_crisis_periods(log_returns, window=20, threshold=2.0):\n",
    "        \"\"\"\n",
    "        Ανιχνεύει περιόδους κρίσης βάσει υψηλής μεταβλητότητας.\n",
    "        \n",
    "        Μέθοδος:\n",
    "        1. Υπολογισμός κυλιόμενης τυπικής απόκλισης (rolling std)\n",
    "        2. Εντοπισμός περιόδων όπου std > threshold * mean(std)\n",
    "        \n",
    "        Στα πραγματικά δεδομένα:\n",
    "        ✓ 2008 Financial Crisis\n",
    "        ✓ 2011 Debt Ceiling Crisis\n",
    "        ✓ 2020 COVID-19 Crash\n",
    "        \"\"\"\n",
    "        \n",
    "        # Κυλιόμενη τυπική απόκλιση (volatility)\n",
    "        log_returns = np.asarray(log_returns).reshape(-1)\n",
    "        rolling_std = pd.Series(log_returns).rolling(window=window).std()\n",
    "        rolling_std_values = rolling_std.values\n",
    "        \n",
    "        # Μέσο επίπεδο volatility\n",
    "        mean_volatility = np.nanmean(rolling_std_values)\n",
    "        std_volatility = np.nanstd(rolling_std_values)\n",
    "        \n",
    "        # Threshold για crisis\n",
    "        crisis_threshold = mean_volatility + threshold * std_volatility\n",
    "        \n",
    "        # Ανίχνευση crisis periods\n",
    "        crisis_mask = rolling_std_values > crisis_threshold\n",
    "        \n",
    "        # Clustering: ομαδοποίηση γειτονικών crisis περιόδων\n",
    "        crisis_periods = []\n",
    "        in_crisis = False\n",
    "        start_idx = None\n",
    "        \n",
    "        for i, is_crisis in enumerate(crisis_mask):\n",
    "            if is_crisis and not in_crisis:\n",
    "                start_idx = i\n",
    "                in_crisis = True\n",
    "            elif not is_crisis and in_crisis:\n",
    "                crisis_periods.append((start_idx, i))\n",
    "                in_crisis = False\n",
    "        \n",
    "        if in_crisis:\n",
    "            crisis_periods.append((start_idx, len(crisis_mask)))\n",
    "        \n",
    "        return {\n",
    "            'log_returns': log_returns,\n",
    "            'rolling_volatility': rolling_std_values,\n",
    "            'mean_volatility': mean_volatility,\n",
    "            'crisis_periods': crisis_periods,\n",
    "            'crisis_threshold': crisis_threshold\n",
    "        }\n",
    "\n",
    "\n",
    "# ============================================================================\n",
    "# ΤΜΗΜΑ 2: ΚΛΑΣΕΙΣ ΜΟΝΤΕΛΩΝ (Ίδιες με πριν)\n",
    "# ============================================================================\n",
    "\n",
    "class DenoisingDataset(Dataset):\n",
    "    def __init__(self, X_noisy, y_clean):\n",
    "        self.X_noisy = torch.FloatTensor(X_noisy)\n",
    "        self.y_clean = torch.FloatTensor(y_clean)\n",
    "    \n",
    "    def __len__(self):\n",
    "        return len(self.X_noisy)\n",
    "    \n",
    "    def __getitem__(self, idx):\n",
    "        x = self.X_noisy[idx].unsqueeze(0)\n",
    "        y = self.y_clean[idx].unsqueeze(0)\n",
    "        return x, y\n",
    "\n",
    "\n",
    "class Conv1DAutoencoder(nn.Module):\n",
    "    def __init__(self, input_length=64, latent_dim=16):\n",
    "        super(Conv1DAutoencoder, self).__init__()\n",
    "        \n",
    "        # Encoder\n",
    "        self.enc_conv1 = nn.Conv1d(1, 32, kernel_size=5, padding=2, stride=1)\n",
    "        self.enc_bn1 = nn.BatchNorm1d(32)\n",
    "        self.enc_pool1 = nn.MaxPool1d(kernel_size=2)\n",
    "        \n",
    "        self.enc_conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2, stride=1)\n",
    "        self.enc_bn2 = nn.BatchNorm1d(64)\n",
    "        self.enc_pool2 = nn.MaxPool1d(kernel_size=2)\n",
    "        \n",
    "        self.enc_conv3 = nn.Conv1d(64, 128, kernel_size=3, padding=1, stride=1)\n",
    "        self.enc_bn3 = nn.BatchNorm1d(128)\n",
    "        \n",
    "        # Bottleneck\n",
    "        bottleneck_size = 128 * (input_length // 4)\n",
    "        self.bottleneck_fc1 = nn.Linear(bottleneck_size, latent_dim)\n",
    "        self.bottleneck_fc2 = nn.Linear(latent_dim, bottleneck_size)\n",
    "        \n",
    "        # Decoder\n",
    "        self.dec_conv1 = nn.Conv1d(128, 128, kernel_size=3, padding=1)\n",
    "        self.dec_bn1 = nn.BatchNorm1d(128)\n",
    "        self.dec_up1 = nn.Upsample(scale_factor=2, mode='nearest')\n",
    "        \n",
    "        self.dec_conv2 = nn.Conv1d(128, 64, kernel_size=5, padding=2)\n",
    "        self.dec_bn2 = nn.BatchNorm1d(64)\n",
    "        self.dec_up2 = nn.Upsample(scale_factor=2, mode='nearest')\n",
    "        \n",
    "        self.dec_conv3 = nn.Conv1d(64, 32, kernel_size=5, padding=2)\n",
    "        self.dec_bn3 = nn.BatchNorm1d(32)\n",
    "        \n",
    "        self.dec_conv4 = nn.Conv1d(32, 1, kernel_size=5, padding=2)\n",
    "    \n",
    "    def encode(self, x):\n",
    "        x = F.relu(self.enc_bn1(self.enc_conv1(x)))\n",
    "        x = self.enc_pool1(x)\n",
    "        x = F.relu(self.enc_bn2(self.enc_conv2(x)))\n",
    "        x = self.enc_pool2(x)\n",
    "        x = F.relu(self.enc_bn3(self.enc_conv3(x)))\n",
    "        x = x.view(x.size(0), -1)\n",
    "        x = F.relu(self.bottleneck_fc1(x))\n",
    "        return x\n",
    "    \n",
    "    def decode(self, z):\n",
    "        x = F.relu(self.bottleneck_fc2(z))\n",
    "        x = x.view(x.size(0), 128, -1)\n",
    "        x = F.relu(self.dec_bn1(self.dec_conv1(x)))\n",
    "        x = self.dec_up1(x)\n",
    "        x = F.relu(self.dec_bn2(self.dec_conv2(x)))\n",
    "        x = self.dec_up2(x)\n",
    "        x = F.relu(self.dec_bn3(self.dec_conv3(x)))\n",
    "        x = self.dec_conv4(x)\n",
    "        return x\n",
    "    \n",
    "    def forward(self, x):\n",
    "        z = self.encode(x)\n",
    "        x_recon = self.decode(z)\n",
    "        return x_recon\n",
    "\n",
    "\n",
    "class TDNN(nn.Module):\n",
    "    def __init__(self, input_length=64, time_delays=(1, 2, 4, 8)):\n",
    "        super(TDNN, self).__init__()\n",
    "        \n",
    "        self.input_length = input_length\n",
    "        self.time_delays = time_delays\n",
    "        n_delayed_features = 1 + len(time_delays)\n",
    "        \n",
    "        self.fc1 = nn.Linear(input_length * n_delayed_features, 128)\n",
    "        self.bn1 = nn.BatchNorm1d(128)\n",
    "        self.dropout1 = nn.Dropout(0.2)\n",
    "        \n",
    "        self.fc2 = nn.Linear(128, 64)\n",
    "        self.bn2 = nn.BatchNorm1d(64)\n",
    "        self.dropout2 = nn.Dropout(0.2)\n",
    "        \n",
    "        self.fc3 = nn.Linear(64, 32)\n",
    "        self.fc4 = nn.Linear(32, input_length)\n",
    "    \n",
    "    def forward(self, x):\n",
    "        batch_size = x.size(0)\n",
    "        x = x.squeeze(1)\n",
    "        \n",
    "        delayed_features = [x]\n",
    "        for delay in self.time_delays:\n",
    "            if delay > 0:\n",
    "                delayed = torch.cat([\n",
    "                    torch.zeros(batch_size, delay, device=x.device),\n",
    "                    x[:, :-delay]\n",
    "                ], dim=1)\n",
    "                delayed_features.append(delayed)\n",
    "        \n",
    "        x_delayed = torch.cat(delayed_features, dim=1)\n",
    "        \n",
    "        x = F.relu(self.bn1(self.fc1(x_delayed)))\n",
    "        x = self.dropout1(x)\n",
    "        x = F.relu(self.bn2(self.fc2(x)))\n",
    "        x = self.dropout2(x)\n",
    "        x = F.relu(self.fc3(x))\n",
    "        x = self.fc4(x)\n",
    "        x = x.unsqueeze(1)\n",
    "        \n",
    "        return x\n",
    "\n",
    "\n",
    "# ============================================================================\n",
    "# ΤΜΗΜΑ 3: ΕΚΠΑΙΔΕΥΣΗ ΚΑΙ ΑΞΙΟΛΟΓΗΣΗ\n",
    "# ============================================================================\n",
    "\n",
    "def create_dataset_from_timeseries(signal_clean, window_size=64, stride=8):\n",
    "    \"\"\"Create sliding window dataset\"\"\"\n",
    "    X, y = [], []\n",
    "    for i in range(0, len(signal_clean) - window_size, stride):\n",
    "        X.append(signal_clean[i:i + window_size])\n",
    "        y.append(signal_clean[i:i + window_size])\n",
    "    return np.array(X), np.array(y)\n",
    "\n",
    "\n",
    "def add_noise_to_signal(signal_clean, snr_db=10):\n",
    "    \"\"\"Add Gaussian noise with specified SNR\"\"\"\n",
    "    signal_power = np.mean(signal_clean ** 2)\n",
    "    noise_power = signal_power / (10 ** (snr_db / 10))\n",
    "    noise = np.random.normal(0, np.sqrt(noise_power), len(signal_clean))\n",
    "    signal_noisy = signal_clean + noise\n",
    "    return signal_noisy, noise\n",
    "\n",
    "\n",
    "def train_model(model, train_loader, val_loader, epochs=30, model_name=\"Model\"):\n",
    "    \"\"\"Train denoising model\"\"\"\n",
    "    \n",
    "    print(f\"\\n{'='*80}\")\n",
    "    print(f\"ΕΚΠΑΙΔΕΥΣΗ ΜΟΝΤΕΛΟΥ: {model_name}\")\n",
    "    print(f\"{'='*80}\")\n",
    "    \n",
    "    model = model.to(device)\n",
    "    optimizer = optim.Adam(model.parameters(), lr=0.001)\n",
    "    criterion = nn.MSELoss()\n",
    "    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', \n",
    "                                                      factor=0.5, patience=5)\n",
    "    \n",
    "    best_val_loss = float('inf')\n",
    "    patience = 10\n",
    "    patience_counter = 0\n",
    "    train_losses = []\n",
    "    val_losses = []\n",
    "    \n",
    "    for epoch in range(epochs):\n",
    "        model.train()\n",
    "        train_loss = 0.0\n",
    "        \n",
    "        for batch_noisy, batch_clean in train_loader:\n",
    "            batch_noisy = batch_noisy.to(device)\n",
    "            batch_clean = batch_clean.to(device)\n",
    "            \n",
    "            optimizer.zero_grad()\n",
    "            output = model(batch_noisy)\n",
    "            loss = criterion(output, batch_clean)\n",
    "            loss.backward()\n",
    "            optimizer.step()\n",
    "            \n",
    "            train_loss += loss.item()\n",
    "        \n",
    "        train_loss /= len(train_loader)\n",
    "        train_losses.append(train_loss)\n",
    "        \n",
    "        # Validation\n",
    "        model.eval()\n",
    "        val_loss = 0.0\n",
    "        \n",
    "        with torch.no_grad():\n",
    "            for batch_noisy, batch_clean in val_loader:\n",
    "                batch_noisy = batch_noisy.to(device)\n",
    "                batch_clean = batch_clean.to(device)\n",
    "                output = model(batch_noisy)\n",
    "                loss = criterion(output, batch_clean)\n",
    "                val_loss += loss.item()\n",
    "        \n",
    "        val_loss /= len(val_loader)\n",
    "        val_losses.append(val_loss)\n",
    "        scheduler.step(val_loss)\n",
    "        \n",
    "        if val_loss < best_val_loss:\n",
    "            best_val_loss = val_loss\n",
    "            patience_counter = 0\n",
    "        else:\n",
    "            patience_counter += 1\n",
    "        \n",
    "        if patience_counter >= patience:\n",
    "            print(f\"Early stopping at epoch {epoch+1}\")\n",
    "            break\n",
    "        \n",
    "        if (epoch + 1) % 10 == 0 or epoch == 0:\n",
    "            print(f\"Epoch {epoch+1}/{epochs} - Train: {train_loss:.6f}, Val: {val_loss:.6f}\")\n",
    "    \n",
    "    return train_losses, val_losses\n",
    "\n",
    "\n",
    "def evaluate_model(model, test_loader, model_name=\"Model\"):\n",
    "    \"\"\"Evaluate model on test set\"\"\"\n",
    "    \n",
    "    print(f\"\\n{'='*60}\")\n",
    "    print(f\"ΑΞΙΟΛΟΓΗΣΗ: {model_name}\")\n",
    "    print(f\"{'='*60}\")\n",
    "    \n",
    "    model.eval()\n",
    "    all_noisy, all_clean, all_denoised = [], [], []\n",
    "    \n",
    "    with torch.no_grad():\n",
    "        for batch_noisy, batch_clean in test_loader:\n",
    "            batch_noisy = batch_noisy.to(device)\n",
    "            batch_clean = batch_clean.to(device)\n",
    "            output = model(batch_noisy)\n",
    "            \n",
    "            all_noisy.append(batch_noisy.cpu().numpy())\n",
    "            all_clean.append(batch_clean.cpu().numpy())\n",
    "            all_denoised.append(output.cpu().numpy())\n",
    "    \n",
    "    X_test_noisy = np.concatenate(all_noisy, axis=0)\n",
    "    X_test_clean = np.concatenate(all_clean, axis=0)\n",
    "    X_test_denoised = np.concatenate(all_denoised, axis=0)\n",
    "    \n",
    "    mse_noisy = np.mean((X_test_noisy - X_test_clean) ** 2)\n",
    "    mse_denoised = np.mean((X_test_denoised - X_test_clean) ** 2)\n",
    "    mae_noisy = np.mean(np.abs(X_test_noisy - X_test_clean))\n",
    "    mae_denoised = np.mean(np.abs(X_test_denoised - X_test_clean))\n",
    "    \n",
    "    signal_power = np.mean(X_test_clean ** 2)\n",
    "    snr_noise = 10 * np.log10(signal_power / mse_noisy + 1e-10)\n",
    "    snr_denoised = 10 * np.log10(signal_power / mse_denoised + 1e-10)\n",
    "    snr_improvement = snr_denoised - snr_noise\n",
    "    \n",
    "    print(f\"MSE (noisy):     {mse_noisy:.6f}\")\n",
    "    print(f\"MSE (denoised):  {mse_denoised:.6f}\")\n",
    "    print(f\"Βελτίωση MSE:    {(1 - mse_denoised/mse_noisy)*100:.2f}%\")\n",
    "    print(f\"\\nSNR (noisy):     {snr_noise:.2f} dB\")\n",
    "    print(f\"SNR (denoised):  {snr_denoised:.2f} dB\")\n",
    "    print(f\"SNR Improvement: {snr_improvement:.2f} dB\")\n",
    "    \n",
    "    return {\n",
    "        'mse_noisy': mse_noisy,\n",
    "        'mse_denoised': mse_denoised,\n",
    "        'mae_noisy': mae_noisy,\n",
    "        'mae_denoised': mae_denoised,\n",
    "        'snr_noisy': snr_noise,\n",
    "        'snr_denoised': snr_denoised,\n",
    "        'snr_improvement': snr_improvement,\n",
    "        'predictions': X_test_denoised\n",
    "    }\n",
    "\n",
    "\n",
    "# ============================================================================\n",
    "# ΤΜΗΜΑ 4: ΚΥΡΙΑ ΕΚΤΕΛΕΣΗ\n",
    "# ============================================================================\n",
    "\n",
    "def main():\n",
    "    \n",
    "    print(\"\\n\" + \"=\"*80)\n",
    "    print(\"ΦΑΣΗ 1: ΚΑΤΕΒΑΣΜΑ ΠΡΑΓΜΑΤΙΚΩΝ ΔΕΔΟΜΕΝΩΝ\")\n",
    "    print(\"=\"*80)\n",
    "    \n",
    "    # Κατέβασμα S&P 500 (5 χρόνια δεδομένων)\n",
    "    print(\"\\n[1.1] Κατέβασμα δεδομένων S&P 500...\")\n",
    "    df_sp500 = FinancialDataDownloader.download_stock_data(\n",
    "        \"^GSPC\", \n",
    "        start_date=\"2019-01-01\",\n",
    "        end_date=\"2024-12-31\"\n",
    "    )\n",
    "    \n",
    "    if df_sp500 is None:\n",
    "        print(\"Σφάλμα κατεβάσματος. Χρήση συνθετικών δεδομένων...\")\n",
    "        # Fallback σε συνθετικά δεδομένα\n",
    "        length = 1250  # ~5 χρόνια trading days\n",
    "        returns = np.random.normal(0.0005, 0.015, length)\n",
    "        log_returns_sp500 = np.cumsum(returns)\n",
    "        crisis_info = {\n",
    "            'log_returns': log_returns_sp500,\n",
    "            'rolling_volatility': np.full(length, 0.015),\n",
    "            'crisis_periods': [(200, 300), (700, 850)]\n",
    "        }\n",
    "    else:\n",
    "        # Υπολογισμός log-returns\n",
    "        log_returns_sp500 = FinancialDataDownloader.calculate_log_returns(df_sp500)\n",
    "        \n",
    "        # Ανίχνευση περιόδων κρίσης\n",
    "        print(\"\\n[1.2] Ανίχνευση περιόδων κρίσης στα δεδομένα...\")\n",
    "        crisis_info = FinancialDataDownloader.identify_crisis_periods(\n",
    "            log_returns_sp500, \n",
    "            window=20, \n",
    "            threshold=2.0\n",
    "        )\n",
    "        \n",
    "        print(f\"✓ {len(df_sp500)} ημέρες καταγραφής\")\n",
    "        print(f\"✓ Μέση ημερήσια volatility: {crisis_info['mean_volatility']:.4f}\")\n",
    "        print(f\"✓ Περιόδους κρίσης ανιχνευμένες: {len(crisis_info['crisis_periods'])}\")\n",
    "        \n",
    "        for i, (start, end) in enumerate(crisis_info['crisis_periods'][:5]):\n",
    "            print(f\"  Κρίση {i+1}: Days {start}-{end} (διάρκεια: {end-start} ημέρες)\")\n",
    "    \n",
    "    # Κατέβασμα Apple (για σύγκριση)\n",
    "    print(\"\\n[1.3] Κατέβασμα δεδομένων Apple (AAPL)...\")\n",
    "    df_apple = FinancialDataDownloader.download_stock_data(\n",
    "        \"AAPL\",\n",
    "        start_date=\"2019-01-01\",\n",
    "        end_date=\"2024-12-31\"\n",
    "    )\n",
    "    \n",
    "    if df_apple is not None:\n",
    "        log_returns_apple = FinancialDataDownloader.calculate_log_returns(df_apple)\n",
    "        print(f\"✓ Μέσο return: {np.mean(log_returns_apple):.4f}\")\n",
    "        print(f\"✓ Volatility: {np.std(log_returns_apple):.4f}\")\n",
    "    else:\n",
    "        log_returns_apple = log_returns_sp500\n",
    "    \n",
    "    # Χρήση S&P 500 ως κύριο σήμα\n",
    "    signal_clean = crisis_info['log_returns']\n",
    "    \n",
    "    print(\"\\n\" + \"=\"*80)\n",
    "    print(\"ΦΑΣΗ 2: ΠΡΟΕΤΟΙΜΑΣΙΑ ΔΕΔΟΜΕΝΩΝ\")\n",
    "    print(\"=\"*80)\n",
    "    \n",
    "    # Normalize\n",
    "    scaler = StandardScaler()\n",
    "    signal_clean = np.asarray(signal_clean).reshape(-1)\n",
    "    if signal_clean.size == 0:\n",
    "        print(\"✗ Empty signal_clean; using synthetic fallback to avoid StandardScaler crash.\")\n",
    "        signal_clean = np.random.normal(0.0, 1.0, 1250)\n",
    "    signal_clean_normalized = scaler.fit_transform(signal_clean.reshape(-1, 1)).flatten()\n",
    "    \n",
    "    # Προσθήκη noise\n",
    "    print(\"\\n[2.1] Προσθήκη Gaussian noise (SNR=15dB)...\")\n",
    "    signal_noisy, noise = add_noise_to_signal(signal_clean_normalized, snr_db=15)\n",
    "    print(f\"✓ Σήμα: μήκος={len(signal_clean)}, range=[{signal_clean.min():.3f}, {signal_clean.max():.3f}]\")\n",
    "    print(f\"✓ Noise power: {np.mean(noise**2):.6f}\")\n",
    "    \n",
    "    # Δημιουργία dataset\n",
    "    print(\"\\n[2.2] Δημιουργία sliding window dataset...\")\n",
    "    window_size = 64\n",
    "    X_clean, y_clean = create_dataset_from_timeseries(signal_clean_normalized, \n",
    "                                                       window_size=window_size, stride=8)\n",
    "    X_noisy, _ = create_dataset_from_timeseries(signal_noisy, \n",
    "                                                window_size=window_size, stride=8)\n",
    "    \n",
    "    # Split\n",
    "    n_train = int(0.6 * len(X_clean))\n",
    "    n_val = int(0.2 * len(X_clean))\n",
    "    \n",
    "    X_train_noisy = X_noisy[:n_train]\n",
    "    X_train_clean = X_clean[:n_train]\n",
    "    X_val_noisy = X_noisy[n_train:n_train+n_val]\n",
    "    X_val_clean = X_clean[n_train:n_train+n_val]\n",
    "    X_test_noisy = X_noisy[n_train+n_val:]\n",
    "    X_test_clean = X_clean[n_train+n_val:]\n",
    "    \n",
    "    print(f\"✓ Train: {len(X_train_clean)}, Val: {len(X_val_clean)}, Test: {len(X_test_clean)}\")\n",
    "    \n",
    "    # DataLoaders\n",
    "    batch_size = 32\n",
    "    train_ds = DenoisingDataset(X_train_noisy, X_train_clean)\n",
    "    val_ds = DenoisingDataset(X_val_noisy, X_val_clean)\n",
    "    test_ds = DenoisingDataset(X_test_noisy, X_test_clean)\n",
    "    \n",
    "    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)\n",
    "    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)\n",
    "    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)\n",
    "    \n",
    "    # ============================================================================\n",
    "    # ΦΑΣΗ 3: ΕΚΠΑΙΔΕΥΣΗ ΜΟΝΤΕΛΩΝ\n",
    "    # ============================================================================\n",
    "    \n",
    "    print(\"\\n\" + \"=\"*80)\n",
    "    print(\"ΦΑΣΗ 3: ΕΚΠΑΙΔΕΥΣΗ ΜΟΝΤΕΛΩΝ\")\n",
    "    print(\"=\"*80)\n",
    "    \n",
    "    # CNN Autoencoder\n",
    "    print(\"\\n[3.1] CNN Autoencoder...\")\n",
    "    model_cnn = Conv1DAutoencoder(input_length=window_size, latent_dim=16)\n",
    "    train_model(model_cnn, train_loader, val_loader, epochs=30, \n",
    "                model_name=\"CNN Autoencoder (Real Data)\")\n",
    "    \n",
    "    # TDNN\n",
    "    print(\"\\n[3.2] TDNN...\")\n",
    "    model_tdnn = TDNN(input_length=window_size, time_delays=(1, 2, 4, 8))\n",
    "    train_model(model_tdnn, train_loader, val_loader, epochs=30, \n",
    "                model_name=\"TDNN (Real Data)\")\n",
    "    \n",
    "    # ============================================================================\n",
    "    # ΦΑΣΗ 4: ΑΞΙΟΛΟΓΗΣΗ\n",
    "    # ============================================================================\n",
    "    \n",
    "    print(\"\\n\" + \"=\"*80)\n",
    "    print(\"ΦΑΣΗ 4: ΑΞΙΟΛΟΓΗΣΗ ΜΟΝΤΕΛΩΝ\")\n",
    "    print(\"=\"*80)\n",
    "    \n",
    "    results_cnn = evaluate_model(model_cnn, test_loader, \"CNN Autoencoder\")\n",
    "    results_tdnn = evaluate_model(model_tdnn, test_loader, \"TDNN\")\n",
    "    \n",
    "    # ============================================================================\n",
    "    # ΦΑΣΗ 5: ΑΝΙΧΝΕΥΣΗ ΚΡΙΣΕΩΝ\n",
    "    # ============================================================================\n",
    "    \n",
    "    print(\"\\n\" + \"=\"*80)\n",
    "    print(\"ΦΑΣΗ 5: ΑΝΙΧΝΕΥΣΗ ΚΡΙΣΕΩΝ ΣΤΑ ΠΡΑΓΜΑΤΙΚΑ ΔΕΔΟΜΕΝΑ\")\n",
    "    print(\"=\"*80)\n",
    "    \n",
    "    # Full signal prediction\n",
    "    model_cnn.eval()\n",
    "    model_tdnn.eval()\n",
    "    \n",
    "    full_pred_cnn = np.zeros_like(signal_noisy)\n",
    "    full_pred_tdnn = np.zeros_like(signal_noisy)\n",
    "    counts = np.zeros_like(signal_noisy)\n",
    "    \n",
    "    print(\"\\n[5.1] Prediction σε ολόκληρο το σήμα...\")\n",
    "    \n",
    "    with torch.no_grad():\n",
    "        for i in range(0, len(signal_noisy) - window_size, 8):\n",
    "            window = signal_noisy[i:i + window_size]\n",
    "            window_tensor = torch.FloatTensor(window).unsqueeze(0).unsqueeze(0).to(device)\n",
    "            \n",
    "            output_cnn = model_cnn(window_tensor).squeeze().cpu().numpy()\n",
    "            output_tdnn = model_tdnn(window_tensor).squeeze().cpu().numpy()\n",
    "            \n",
    "            full_pred_cnn[i:i + window_size] += output_cnn\n",
    "            full_pred_tdnn[i:i + window_size] += output_tdnn\n",
    "            counts[i:i + window_size] += 1\n",
    "    \n",
    "    full_pred_cnn = np.divide(full_pred_cnn, counts, where=counts > 0, out=full_pred_cnn)\n",
    "    full_pred_tdnn = np.divide(full_pred_tdnn, counts, where=counts > 0, out=full_pred_tdnn)\n",
    "    \n",
    "    print(\"✓ Prediction completed\")\n",
    "    \n",
    "    # Analyze crisis periods\n",
    "    print(\"\\n[5.2] Ανάλυση περιόδων κρίσης...\")\n",
    "    \n",
    "    if 'crisis_periods' in crisis_info:\n",
    "        for i, (start, end) in enumerate(crisis_info['crisis_periods'][:3]):\n",
    "            crisis_window = signal_clean_normalized[start:end]\n",
    "            noisy_window = signal_noisy[start:end]\n",
    "            denoised_window = full_pred_cnn[start:end]\n",
    "            \n",
    "            mse_before = np.mean((noisy_window - crisis_window) ** 2)\n",
    "            mse_after = np.mean((denoised_window - crisis_window) ** 2)\n",
    "            improvement = (1 - mse_after/mse_before) * 100\n",
    "            \n",
    "            print(f\"\\n  Κρίση {i+1} (Days {start}-{end}):\")\n",
    "            print(f\"    Original volatility: {np.std(crisis_window):.4f}\")\n",
    "            print(f\"    Noisy volatility: {np.std(noisy_window):.4f}\")\n",
    "            print(f\"    Denoised volatility: {np.std(denoised_window):.4f}\")\n",
    "            print(f\"    MSE improvement: {improvement:.1f}%\")\n",
    "    \n",
    "    # ============================================================================\n",
    "    # ΦΑΣΗ 6: ΑΠΟΘΗΚΕΥΣΗ ΑΠΟΤΕΛΕΣΜΑΤΩΝ\n",
    "    # ============================================================================\n",
    "    \n",
    "    print(\"\\n\" + \"=\"*80)\n",
    "    print(\"ΦΑΣΗ 6: ΑΠΟΘΗΚΕΥΣΗ ΑΠΟΤΕΛΕΣΜΑΤΩΝ\")\n",
    "    print(\"=\"*80)\n",
    "    \n",
    "    np.savez('/home/claude/denoising_results_real_data.npz',\n",
    "             signal_clean=signal_clean_normalized,\n",
    "             signal_noisy=signal_noisy,\n",
    "             signal_denoised_cnn=full_pred_cnn,\n",
    "             signal_denoised_tdnn=full_pred_tdnn,\n",
    "             crisis_periods=np.array(crisis_info.get('crisis_periods', [])),\n",
    "             log_returns_original=crisis_info['log_returns'])\n",
    "    \n",
    "    print(\"✓ Αποτελέσματα αποθηκεύτηκαν στο denoising_results_real_data.npz\")\n",
    "    \n",
    "    # Summary\n",
    "    print(\"\\n\" + \"=\"*80)\n",
    "    print(\"ΣΥΝΟΨΗ ΑΠΟΤΕΛΕΣΜΑΤΩΝ (ΠΡΑΓΜΑΤΙΚΑ ΔΕΔΟΜΕΝΑ)\")\n",
    "    print(\"=\"*80)\n",
    "    \n",
    "    print(f\"\\nCNN Autoencoder:\")\n",
    "    print(f\"  SNR Improvement: {results_cnn['snr_improvement']:.2f} dB\")\n",
    "    print(f\"  MSE Reduction: {(1-results_cnn['mse_denoised']/results_cnn['mse_noisy'])*100:.1f}%\")\n",
    "    \n",
    "    print(f\"\\nTDNN:\")\n",
    "    print(f\"  SNR Improvement: {results_tdnn['snr_improvement']:.2f} dB\")\n",
    "    print(f\"  MSE Reduction: {(1-results_tdnn['mse_denoised']/results_tdnn['mse_noisy'])*100:.1f}%\")\n",
    "    \n",
    "    best_model = \"TDNN\" if results_tdnn['snr_improvement'] > results_cnn['snr_improvement'] else \"CNN\"\n",
    "    print(f\"\\n➜ Καλύτερο μοντέλο: {best_model}\")\n",
    "    \n",
    "    return {\n",
    "        'signal_clean': signal_clean_normalized,\n",
    "        'signal_noisy': signal_noisy,\n",
    "        'signal_denoised_cnn': full_pred_cnn,\n",
    "        'signal_denoised_tdnn': full_pred_tdnn,\n",
    "        'results_cnn': results_cnn,\n",
    "        'results_tdnn': results_tdnn,\n",
    "        'crisis_periods': crisis_info.get('crisis_periods', []),\n",
    "        'df_sp500': df_sp500,\n",
    "        'df_apple': df_apple if df_apple is not None else None\n",
    "    }\n",
    "\n",
    "\n",
    "if __name__ == \"__main__\":\n",
    "    results = main()\n",
    "    print(\"\\n\" + \"=\"*80)\n",
    "    print(\"ΟΛΟΚΛΗΡΩΣΗ ΑΝΑΛΥΣΗΣ\")\n",
    "    print(\"=\"*80)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "b465d085",
   "metadata": {},
   "outputs": [],
   "source": []
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "pytorch_env",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.12.5"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}


Using device: cuda
TIME SERIES DENOISING WITH PYTORCH - REAL FINANCIAL DATA

ΦΑΣΗ 1: ΚΑΤΕΒΑΣΜΑ ΠΡΑΓΜΑΤΙΚΩΝ ΔΕΔΟΜΕΝΩΝ

[1.1] Κατέβασμα δεδομένων S&P 500...

[📊] Κατέβασμα δεδομένων για ^GSPC (2019-01-01 → 2024-12-31)...
✓ 1509 κλεισίματα κατεβασμένα για ^GSPC

[1.2] Ανίχνευση περιόδων κρίσης στα δεδομένα...
✓ 1509 ημέρες καταγραφής
✓ Μέση ημερήσια volatility: nan
✓ Περιόδους κρίσης ανιχνευμένες: 0

[1.3] Κατέβασμα δεδομένων Apple (AAPL)...

[📊] Κατέβασμα δεδομένων για AAPL (2019-01-01 → 2024-12-31)...
✓ 1509 κλεισίματα κατεβασμένα για AAPL
✓ Μέσο return: nan
✓ Volatility: nan

ΦΑΣΗ 2: ΠΡΟΕΤΟΙΜΑΣΙΑ ΔΕΔΟΜΕΝΩΝ


ValueError: Found array with 0 sample(s) (shape=(0, 1)) while a minimum of 1 is required by StandardScaler.